# BioForge - Complete Pipeline Notebook
Run each cell top-to-bottom. Each code cell loads a module into memory;
**EXECUTE** cells actually run the phase.

---

##  Setup - Run once per session

In [ ]:
# S1 - Install all dependencies
!pip install biopython networkx requests torch cobra pymoo matplotlib pandas

In [ ]:
# S2 - Mount Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/bio_pipeline')  # ← change if needed
print('Working directory:', os.getcwd())

In [ ]:
# S3 - Clone ProteinMPNN (Phase 2 needs this)
import os
if not os.path.isdir('./ProteinMPNN'):
    !git clone --depth 1 https://github.com/dauparas/ProteinMPNN.git ./ProteinMPNN
else:
    print('ProteinMPNN already present.')

In [ ]:
# S4 - Download iML1515 metabolic model (Phase 3 needs this)
import os
os.makedirs('./models', exist_ok=True)
if not os.path.exists('./models/iML1515.xml'):
    !wget -q -O ./models/iML1515.xml http://bigg.ucsd.edu/static/models/iML1515.xml
    print('Downloaded iML1515.xml')
else:
    print('iML1515.xml already present.')

---
## Phase 1 - Structural Ingestion & Metabolic Graph
Run all 4 cells below in order.

### `Phase 1 · structure_parser.py`

In [ ]:
import os
from typing import Tuple
from Bio.PDB import PDBList, PDBParser
import torch

class StructureParser:
    def __init__(self, pdb_code: str, storage_dir: str = "./pdb_cache"):
        self.pdb_code = pdb_code.lower()
        self.storage_dir = storage_dir
        os.makedirs(self.storage_dir, exist_ok=True)
        
    def fetch_pdb_file(self) -> str:
        """Downloads the raw PDB file from the RCSB server with strict verification."""
        print(f"[INFO] Fetching PDB file for {self.pdb_code.upper()} from RCSB...")
        pdbl = PDBList()
        file_path = pdbl.retrieve_pdb_file(self.pdb_code, pdir=self.storage_dir, file_format="pdb")
        
        # Check for network-failure resilience or empty file drops
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"[ERROR] PDB file failed to download or save at: {file_path}")
        if os.path.getsize(file_path) < 1024:  # Must be greater than 1KB
            raise IOError(f"[ERROR] Downloaded PDB file is corrupt or empty (Size: {os.path.getsize(file_path)} bytes).")
            
        return file_path

    def extract_backbone_tensors(self, file_path: str, chain_id: str = "A") -> Tuple[torch.Tensor, str]:
        """
        Parses PDB geometry and extracts a tensor of shape [L, 4, 3] 
        and the accompanying sequence string.
        """
        print(f"[INFO] Extracting backbone coordinates from Chain {chain_id}...")
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(self.pdb_code, file_path)
        model = structure[0]
        
        if chain_id not in model:
            available_chains = [c.id for c in model.get_chains()]
            raise KeyError(f"Chain {chain_id} not found in PDB. Available chains: {available_chains}")
            
        chain = model[chain_id]
        backbone_coordinates = []
        residue_sequence = []
        
        three_to_one = {
            'ALA':'A', 'CYS':'C', 'ASP':'D', 'GLU':'E', 'PHE':'F',
            'GLY':'G', 'HIS':'H', 'ILE':'I', 'LYS':'K', 'LEU':'L',
            'MET':'M', 'ASN':'N', 'PRO':'P', 'GLN':'Q', 'ARG':'R',
            'SER':'S', 'THR':'T', 'VAL':'V', 'TRP':'W', 'TYR':'Y'
        }

        for residue in chain:
            if residue.id[0] != " ":
                continue
                
            res_name = residue.get_resname().strip()
            if res_name not in three_to_one:
                continue
                
            required_atoms = ["N", "CA", "C", "O"]
            if all(atom in residue for atom in required_atoms):
                residue_coords = [residue[atom].get_coord() for atom in required_atoms]
                backbone_coordinates.append(residue_coords)
                residue_sequence.append(three_to_one[res_name])
                
        if not backbone_coordinates:
            raise ValueError(f"[ERROR] No valid structural residues extracted from Chain {chain_id}.")
            
        coordinate_tensor = torch.tensor(backbone_coordinates, dtype=torch.float32)
        sequence_str = "".join(residue_sequence)
        
        print(f"[SUCCESS] Ingested Sequence: {sequence_str}")
        print(f"[SUCCESS] Coordinates Matrix Shape: {list(coordinate_tensor.shape)}")
        
        return coordinate_tensor, sequence_str

### `Phase 1 · metabolic_miner.py`

In [ ]:
"""
metabolic_miner.py
-------------------
Phase 1.2: Metabolic Database Mining via the KEGG REST API.

CHANGELOG FROM PRIOR VERSION
=============================
1. FIXED: aa_to_cpd had two wrong compound IDs (silent data-corruption bug).

   - 'D' (Aspartate) pointed to cpd:C00042, which is KEGG's ID for
     SUCCINATE, not aspartate. Every 'D' residue was silently pulling
     succinate pathway/enzyme data instead of aspartate's. The correct
     ID, verified against KEGG's own /list/compound dump, is C00049.

   - 'K' (Lysine) pointed to cpd:C00408, which is not lysine at all.
     The correct ID is C00047 (confirmed: "C00047 for L-lysine" per
     KEGG's own compound database documentation).

   Both were "valid" compound IDs, so the original code never raised
   an exception or triggered the fallback path -- it just returned
   confidently wrong data. This is the most dangerous kind of bug in
   this pipeline, because Phase 3's COBRApy flux model and Phase 4's
   NSGA-II loop have no way to know the enzyme/pathway data they're
   optimizing against is attached to the wrong molecule.

2. FIXED: Pathway selection used eco_pathways[0] with no filtering.

   KEGG's /link/pathway/cpd:XXXXX endpoint returns pathway IDs in
   whatever order KEGG's database emits them -- NOT ranked by
   specificity. A compound like aspartate links to a dozen pathways,
   including broad "overview" maps (e.g. eco01100 Metabolic pathways,
   eco01230 Biosynthesis of amino acids) that carry hundreds of
   unrelated enzymes. If one of those overview IDs happened to sort
   first, associated_enzymes would fill up with irrelevant genome-wide
   markers instead of the specific biosynthesis pathway you actually
   want.

   Fix: explicitly exclude known KEGG overview/global pathway IDs
   before selecting a target pathway, so we land on the specific
   pathway (e.g. "Lysine biosynthesis" rather than "Metabolic
   pathways").

3. KEPT from previous hardening pass: on-disk JSON caching, source
   provenance tagging ("kegg" vs "fallback"), 10s request timeouts,
   and a short polite delay between calls.

4. ADDED: try/except around cache load, so a partially-written or
   corrupted cache file (e.g. from a killed process) doesn't crash
   every subsequent run -- it just rebuilds the cache from scratch.

5. FIXED (this pass): /link/pathway/cpd:CXXXXX never returns organism-
   scoped IDs. It only returns generic REFERENCE pathway maps, formatted
   as "path:mapXXXXX" -- confirmed against KEGG usage examples showing
   e.g. cpd:C00001 -> "path:map00190". The prior filter looked for the
   substring "path:eco" in that response, which structurally never
   appears there, so every single lookup fell through to the fallback
   path regardless of whether KEGG had data or not.

   Fix: KEGG reuses the same 5-digit pathway number across the generic
   map and every organism-specific version of it (map00270 / eco00270
   / hsa00270 are the same pathway, just genome-scoped differently).
   So we take the returned "map#####" ID and swap the "map" prefix for
   "eco" directly -- no extra API round trip needed. The overview-
   pathway exclusion list still applies after the swap, since generic
   maps like eco01100 can still appear post-conversion.
"""

import os
import json
import time
import requests


# KEGG pathway IDs that are broad "overview" maps rather than specific
# biosynthesis pathways. These get excluded when picking a target
GENERIC_OVERVIEW_PATHWAYS = {
    "eco01100",  # Metabolic pathways (global overview)
    "eco01110",  # Biosynthesis of secondary metabolites
    "eco01120",  # Microbial metabolism in diverse environments
    "eco01200",  # Carbon metabolism
    "eco01210",  # 2-Oxocarboxylic acid metabolism
    "eco01230",  # Biosynthesis of amino acids (overview, not residue-specific)
    "eco01240",  # Biosynthesis of cofactors
    "eco01250",  # Biosynthesis of nucleotide sugars
}


class MetabolicMiner:
    def __init__(self, cache_file: str = "kegg_cache.json"):
        self.kegg_base_url = "https://rest.kegg.jp"
        self.cache_file = cache_file
        self.cache = self._load_cache()

        # Verified against KEGG's live /list/compound and /find/compound
        # endpoints. Each entry below was cross-checked individually.
        self.aa_to_cpd = {
            'A': 'cpd:C00041',  # L-Alanine
            'R': 'cpd:C00062',  # L-Arginine
            'N': 'cpd:C00152',  # L-Asparagine
            'D': 'cpd:C00049',  # L-Aspartate  (FIXED: was C00042 / Succinate)
            'C': 'cpd:C00097',  # L-Cysteine
            'Q': 'cpd:C00064',  # L-Glutamine
            'E': 'cpd:C00025',  # L-Glutamate
            'G': 'cpd:C00037',  # Glycine
            'H': 'cpd:C00135',  # L-Histidine
            'I': 'cpd:C00407',  # L-Isoleucine
            'L': 'cpd:C00123',  # L-Leucine
            'K': 'cpd:C00047',  # L-Lysine    (FIXED: was C00408 / wrong compound)
            'M': 'cpd:C00073',  # L-Methionine
            'F': 'cpd:C00079',  # L-Phenylalanine
            'P': 'cpd:C00148',  # L-Proline
            'S': 'cpd:C00065',  # L-Serine
            'T': 'cpd:C00188',  # L-Threonine
            'W': 'cpd:C00078',  # L-Tryptophan
            'Y': 'cpd:C00082',  # L-Tyrosine
            'V': 'cpd:C00183',  # L-Valine
        }

    def _load_cache(self) -> dict:
        """Load on-disk cache, tolerating a missing or corrupted file."""
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'r') as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError) as e:
                print(f"[WARNING] Cache file unreadable ({e}); rebuilding cache from scratch.")
                return {}
        return {}

    def _save_cache(self):
        try:
            with open(self.cache_file, 'w') as f:
                json.dump(self.cache, f, indent=4)
        except OSError as e:
            print(f"[WARNING] Could not persist cache to disk: {e}")

    def _select_specific_pathway(self, eco_pathways: list) -> str:
        """
        Given a list of path:ecoXXXXX IDs linked to a compound, return the
        most specific (non-overview) pathway. Falls back to the first
        entry only if every candidate is a generic overview pathway.
        """
        specific = [
            p for p in eco_pathways
            if p.replace("path:", "") not in GENERIC_OVERVIEW_PATHWAYS
        ]
        if specific:
            return specific[0]
        # Every linked pathway was a generic overview map -- still return
        # something rather than failing, but this is a weaker signal.
        return eco_pathways[0]

    def fetch_ec_pathway_data(self, amino_acid: str) -> dict:
        """Queries KEGG using compound-to-pathway/enzyme linkage strategies with caching."""
        aa_upper = amino_acid.upper()
        if aa_upper not in self.aa_to_cpd:
            return self._get_fallback_stoichiometry(aa_upper, reason="Invalid residue code")

        if aa_upper in self.cache:
            return self.cache[aa_upper]

        cpd_id = self.aa_to_cpd[aa_upper]

        try:
            pathway_url = f"{self.kegg_base_url}/link/pathway/{cpd_id}"
            print(f"[API CALL] Querying pathway links for {aa_upper} ({cpd_id})...")
            p_response = requests.get(pathway_url, timeout=10)
            time.sleep(0.1)

            if p_response.status_code != 200 or not p_response.text.strip():
                raise ConnectionError("Empty or invalid response from KEGG pathway link.")

            # FIXED: this endpoint only ever returns generic "path:mapXXXXX"
            # reference IDs (e.g. cpd:C00001 -> "path:map00190"), never
            eco_pathways = []
            for line in p_response.text.strip().split("\n"):
                parts = line.split("\t")
                if len(parts) == 2 and "path:map" in parts[1]:
                    map_id = parts[1].strip()
                    eco_id = map_id.replace("path:map", "path:eco")
                    eco_pathways.append(eco_id)

            if not eco_pathways:
                raise ValueError(f"No pathway links returned by KEGG for compound {cpd_id}")

            # FIXED: was eco_pathways[0] with no filtering -- now excludes
            # generic overview pathways so we land on the residue-specific
            target_pathway = self._select_specific_pathway(eco_pathways)

            enzyme_url = f"{self.kegg_base_url}/link/enzyme/{target_pathway.replace('path:', '')}"
            print(f"[API CALL] Querying actual host enzyme markers for {target_pathway}...")
            e_response = requests.get(enzyme_url, timeout=10)
            time.sleep(0.1)

            # --- TEMPORARY DEBUG: remove once we've diagnosed the empty-enzyme-list issue ---
            print(f"[DEBUG] Enzyme URL: {enzyme_url}")
            print(f"[DEBUG] Status: {e_response.status_code} | Body (first 300 chars): {e_response.text[:300]!r}")
            # --- END TEMPORARY DEBUG ---

            associated_enzymes = []
            if e_response.status_code == 200 and e_response.text.strip():
                for line in e_response.text.strip().split("\n"):
                    parts = line.split("\t")
                    if len(parts) == 2:
                        associated_enzymes.append(parts[1].replace("ec:", "").strip())

            result = {
                "amino_acid": aa_upper,
                "pathway_id": target_pathway,
                "associated_enzymes": associated_enzymes[:5],
                "stoichiometry": self._get_fallback_stoichiometry(aa_upper)["stoichiometry"],
                "provenance": "kegg"
            }

            self.cache[aa_upper] = result
            self._save_cache()
            return result

        except Exception as e:
            print(f"[WARNING] KEGG query anomaly for {aa_upper}: {str(e)}. Executing fallback framework.")
            return self._get_fallback_stoichiometry(aa_upper, reason=str(e))

    def _get_fallback_stoichiometry(self, aa: str, reason: str = "none") -> dict:
        atp_costs = {
            'A': 11.7, 'R': 27.3, 'N': 14.7, 'D': 12.7, 'C': 24.7,
            'Q': 16.3, 'E': 12.3, 'G': 11.7, 'H': 38.3, 'I': 32.3,
            'L': 27.3, 'K': 30.3, 'M': 34.3, 'F': 39.0, 'P': 20.3,
            'S': 11.7, 'T': 18.7, 'W': 60.3, 'Y': 50.0, 'V': 23.3
        }
        return {
            "amino_acid": aa,
            "pathway_id": "unknown_fallback",
            "associated_enzymes": [],
            "stoichiometry": {"atp_cost": atp_costs.get(aa, 20.0), "carbon_precursor": "Glucose"},
            "provenance": "fallback",
            "fallback_reason": reason
        }


### `Phase 1 · graph_compiler.py`

In [ ]:
import networkx as nx

class ManufacturingGraphCompiler:
    def __init__(self):
        self.graph = nx.DiGraph()

    def build_expression_dag(self, sequence: str, metabolic_data_pool: dict) -> nx.DiGraph:
        """Compiles sequence parameters and pathway attributes into a Directed Acyclic Graph."""
        print("[INFO] Initiating Construction of NetworkX Manufacturing Graph...")
        self.graph.clear() # Clear state
        
        self.graph.add_node("E_coli_Biomass_Sink", type="Output", capacity="Target_Expression")
        unique_residues = set(sequence)
        
        for residue in unique_residues:
            data = metabolic_data_pool.get(residue, {})
            atp_weight = data.get("stoichiometry", {}).get("atp_cost", 20.0)
            precursor = data.get("stoichiometry", {}).get("carbon_precursor", "Glucose")
            provenance = data.get("provenance", "unknown")
            
            amino_acid_node = f"Residue_{residue}"
            precursor_node = f"Feedstock_{precursor}"
            
            self.graph.add_node(amino_acid_node, type="Amino_Acid_Pool", residue=residue, provenance=provenance)
            self.graph.add_node(precursor_node, type="Raw_Input", cost_index=1.0)
            
            self.graph.add_edge(
                precursor_node, 
                amino_acid_node, 
                weight=atp_weight, 
                active_enzymes=data.get("associated_enzymes", []),
                provenance=provenance
            )
            
            seq_frequency = sequence.count(residue)
            self.graph.add_edge(amino_acid_node, "E_coli_Biomass_Sink", weight=0.0, total_required_units=seq_frequency)
            
        print(f"[SUCCESS] Graph Compiled. Active Nodes: {self.graph.number_of_nodes()}, Directed Edges: {self.graph.number_of_edges()}")
        return self.graph

### `Phase 1 · run_phase1.py    EXECUTE`

In [ ]:
# run_phase1.py
"""
Phase 1 Orchestrator.

CHANGE FROM PRIOR VERSION
==========================
Previously this script only printed an audit block and held all outputs
in memory (coordinate_tensor, cleaned_sequence, production_graph never
touched disk). Phase 2 has no way to consume in-memory Python objects
from a separate script/session, so this version adds the persistence
step we locked in as the Phase 1 -> Phase 2 handoff contract:

  ./pipeline_artifacts/phase1_structure.pt   -- torch.save({'bb_coords': tensor})
  ./pipeline_artifacts/phase1_manifest.json  -- sidecar metadata

Everything else (structure parsing, metabolic mining, graph compilation)
is unchanged.
"""

import os
import json
from datetime import datetime, timezone

import torch


# Fixed per the locked handoff contract -- change here if the target
# PDB/chain ever changes, nowhere else.
PDB_CODE = "1zfu"
CHAIN_ID = "A"
ARTIFACTS_DIR = "./pipeline_artifacts"


def persist_phase1_outputs(coordinate_tensor: torch.Tensor, sequence: str, raw_pdb_path: str) -> None:
    """
    Writes the dual-file handoff contract Phase 2 depends on:
      1. phase1_structure.pt  -- {'bb_coords': Tensor[L, 4, 3]}
      2. phase1_manifest.json -- human-readable sidecar, cross-checkable
         against the tensor before Phase 2 trusts it.

    NOTE: raw_pdb_path is included because ProteinMPNN's own PDB parser
    needs the actual .pdb/.ent file on disk -- our extracted tensor isn't
    a substitute input for their pipeline. Without this, Phase 2 would
    have no reliable way to locate the source structure file.
    """
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)

    coord_file = "phase1_structure.pt"
    coord_path = os.path.join(ARTIFACTS_DIR, coord_file)
    torch.save({"bb_coords": coordinate_tensor}, coord_path)

    manifest = {
        "pdb_id": PDB_CODE.upper(),
        "chain_id": CHAIN_ID,
        "sequence": sequence,
        "num_residues": len(sequence),
        "coord_file": coord_file,
        "raw_pdb_path": os.path.abspath(raw_pdb_path),
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    manifest_path = os.path.join(ARTIFACTS_DIR, "phase1_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"[SUCCESS] Persisted Phase 1 handoff artifacts to '{ARTIFACTS_DIR}/':")
    print(f"           - {coord_file}  (tensor shape {list(coordinate_tensor.shape)})")
    print(f"           - phase1_manifest.json")


def execute_pipeline_phase_1():
    print("================================================================================")
    print("      PRODUCTION RUN: PHASE 1 DATA COUPLING HARDENED INITIALIZATION")
    print("================================================================================")

    # 0. Wipe all cached artifacts from any previous run so no stale
    #    manifest, PDB file, or mask can bleed into this pipeline.
    import shutil
    _CLEAN_DIRS = ["./pipeline_artifacts", "./pdb_cache", "./phase2_work"]
    for _d in _CLEAN_DIRS:
        if os.path.exists(_d):
            shutil.rmtree(_d)
            print(f"[CLEAN] Removed stale directory: {_d}")
        else:
            print(f"[CLEAN] Already clean: {_d}")
    print("[CLEAN]  Fresh slate - all prior run artifacts cleared.\n")

    # 1. Parse and download structural metrics safely
    parser = StructureParser(pdb_code=PDB_CODE)
    file_path = parser.fetch_pdb_file()
    coordinate_tensor, cleaned_sequence = parser.extract_backbone_tensors(file_path, chain_id=CHAIN_ID)

    # 2. Query KEGG database using link mapping and persistent cache
    miner = MetabolicMiner()
    metabolic_data_pool = {}
    print("\n[START] Executing Cached / Active Query Sequences across KEGG Links...")
    for residue in set(cleaned_sequence):
        pathway_metadata = miner.fetch_ec_pathway_data(residue)
        metabolic_data_pool[residue] = pathway_metadata

    # 3. Assemble structural network topology model
    compiler = ManufacturingGraphCompiler()
    production_graph = compiler.build_expression_dag(cleaned_sequence, metabolic_data_pool)

    # 4. Persist the handoff artifacts Phase 2 depends on
    print()
    persist_phase1_outputs(coordinate_tensor, cleaned_sequence, raw_pdb_path=file_path)

    # 5. Infrastructure Provenance and Integrity Verification Audit
    print("\n================================================================================")
    print("                PHASE 1 PIPELINE AUDIT & INTEGRITY REVIEWS")
    print("================================================================================")
    kegg_sourced = sum(1 for res in metabolic_data_pool.values() if res["provenance"] == "kegg")
    fallback_sourced = sum(1 for res in metabolic_data_pool.values() if res["provenance"] == "fallback")

    print(f"* Structural Chain Target Length: {len(cleaned_sequence)} residues.")
    print(f"* PyTorch Geometric Coordinates:   Tensor matrix spatial dimension {list(coordinate_tensor.shape)}")
    print(f"* Live API Provenance Registry:   {kegg_sourced} Sourced from KEGG | {fallback_sourced} Used Fallback Parameters")

    sample_residue = list(metabolic_data_pool.keys())[0]
    print(f"* Sample Data Mapping [{sample_residue}]: Path -> {metabolic_data_pool[sample_residue]['pathway_id']} | Verified Enzymes Mapped: {metabolic_data_pool[sample_residue]['associated_enzymes']}")
    print("================================================================================\n")



In [ ]:
#  RUN
execute_pipeline_phase_1()

---
## Phase 2 - ProteinMPNN Masked Sequence Inversion
Run all 4 cells below in order.  Requires ProteinMPNN (S3).

### `Phase 2 · mask_builder.py`

In [ ]:
"""
mask_builder.py
----------------
Phase 2.1: Structural Design Mask Construction.

Defines the position-level design mask that gates ProteinMPNN sequence
generation. Built from the locked, audited biological rationale: 8
disulfide-bonded cysteines, the KGRGK receptor-binding patch + C-terminal
aromatic anchor, and local geometry anchors around the alpha-helix
initiation and the P31 turn.

This module has NO dependency on ProteinMPNN itself -- it only needs
Phase 1's manifest (specifically the sequence string) to build and
validate the mask. That makes it independently testable before any
model weights or inference code is involved.
"""

import os
import json


# LOCKED BIOLOGICAL MASK DEFINITION (0-indexed, verified against the real
# 1ZFU sequence: GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY)

EXPECTED_SEQUENCE = "GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY"

# Layer 1: Disulfide Fold Lock - all 3 disulfide pairs (6 Cys), preserves
# the cystine-stabilised alpha/beta (CSalphabeta) fold of Plectasin.
LAYER_1_DISULFIDE_LOCK = {
    3: "C", 14: "C", 18: "C", 29: "C", 36: "C", 38: "C",
}

# Layer 2: Solvent-Exposed Target Patch - F2, D12, Y29, Y40 form the
# lipid-II / PG-binding face critical for antimicrobial activity.
LAYER_2_TARGET_PATCH = {
    1: "F", 11: "D", 28: "Y", 39: "Y",
}

# Layer 3: Local Geometry Anchors - Y25, A31, G33 control beta-hairpin
# geometry and are essential for maintaining the defensin-like backbone.
LAYER_3_GEOMETRY_ANCHORS = {
    24: "Y", 30: "A", 32: "G",
}

FROZEN_LAYERS = {
    "disulfide_fold_lock": LAYER_1_DISULFIDE_LOCK,
    "target_patch": LAYER_2_TARGET_PATCH,
    "geometry_anchors": LAYER_3_GEOMETRY_ANCHORS,
}


class MaskBuilderError(Exception):
    """Raised when the mask fails validation against the actual sequence."""
    pass


class MaskBuilder:
    def __init__(self, artifacts_dir: str = "./pipeline_artifacts"):
        self.artifacts_dir = artifacts_dir

    def _load_manifest(self) -> dict:
        manifest_path = os.path.join(self.artifacts_dir, "phase1_manifest.json")
        if not os.path.exists(manifest_path):
            raise FileNotFoundError(
                f"[ERROR] Phase 1 manifest not found at {manifest_path}. "
                f"Run Phase 1 first."
            )
        with open(manifest_path, "r") as f:
            return json.load(f)

    def _flatten_frozen_indices(self) -> dict:
        """Merge all three layers into a single {index: expected_residue} map,
        raising if any layer accidentally overlaps another."""
        flattened = {}
        for layer_name, layer_indices in FROZEN_LAYERS.items():
            for idx, expected_res in layer_indices.items():
                if idx in flattened:
                    raise MaskBuilderError(
                        f"[ERROR] Index {idx} appears in more than one frozen "
                        f"layer (last seen in '{layer_name}'). Layers must be "
                        f"mutually exclusive."
                    )
                flattened[idx] = expected_res
        return flattened

    def build_mask(self, sequence: str = None) -> dict:
        """
        Builds and validates the position-level mask against the given
        sequence (or the Phase 1 manifest's sequence if none supplied).

        Returns a dict keyed by 0-indexed position string, e.g.:
          {
            "pos_0": {"status": "MUTABLE", "wt": "M", "designable": True},
            "pos_1": {"status": "FROZEN",  "wt": "C", "designable": False,
                       "layer": "disulfide_fold_lock"},
            ...
          }
        """
        if sequence is None:
            manifest = self._load_manifest()
            sequence = manifest["sequence"]

        if sequence != EXPECTED_SEQUENCE:
            raise MaskBuilderError(
                f"[ERROR] Sequence mismatch. Mask was audited against:\n"
                f"  {EXPECTED_SEQUENCE}\n"
                f"but received:\n"
                f"  {sequence}\n"
                f"The frozen-position rationale (cysteines, binding patch, "
                f"geometry anchors) is only valid for the audited sequence. "
                f"Re-audit the mask before proceeding with a different target."
            )

        frozen_map = self._flatten_frozen_indices()

        # Reverse lookup: index -> layer name, for annotation in the output.
        index_to_layer = {}
        for layer_name, layer_indices in FROZEN_LAYERS.items():
            for idx in layer_indices:
                index_to_layer[idx] = layer_name

        mask = {}
        for i, residue in enumerate(sequence):
            if i in frozen_map:
                expected_res = frozen_map[i]
                if residue != expected_res:
                    raise MaskBuilderError(
                        f"[ERROR] Position {i} expected residue '{expected_res}' "
                        f"(per audited mask) but sequence has '{residue}'. "
                        f"Aborting -- freezing the wrong residue would silently "
                        f"corrupt the design space."
                    )
                mask[f"pos_{i}"] = {
                    "status": "FROZEN",
                    "wt": residue,
                    "designable": False,
                    "layer": index_to_layer[i],
                }
            else:
                mask[f"pos_{i}"] = {
                    "status": "MUTABLE",
                    "wt": residue,
                    "designable": True,
                }

        self._validate_mask_summary(mask)
        return mask

    def _validate_mask_summary(self, mask: dict) -> None:
        frozen_count = sum(1 for v in mask.values() if v["status"] == "FROZEN")
        mutable_count = sum(1 for v in mask.values() if v["status"] == "MUTABLE")

        # Plectasin (1ZFU): 40 residues, 13 frozen, 27 mutable
        if frozen_count != 13:
            raise MaskBuilderError(
                f"[ERROR] Expected 13 frozen positions, got {frozen_count}. "
                f"Mask does not match the audited spec."
            )
        if mutable_count != 27:
            raise MaskBuilderError(
                f"[ERROR] Expected 27 mutable positions, got {mutable_count}. "
                f"Mask does not match the audited spec."
            )

    def save_mask(self, mask: dict, output_path: str = None) -> str:
        if output_path is None:
            output_path = os.path.join(self.artifacts_dir, "phase2_mask.json")
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(mask, f, indent=2)
        return output_path



### `Phase 2 · protein_mpnn_runner.py`

In [ ]:
"""
protein_mpnn_runner.py
----
Phase 2.2: ProteinMPNN Inference.

IMPORTANT -- READ BEFORE RUNNING
==================================
This module orchestrates ProteinMPNN via its own documented command-line
scripts (subprocess calls) rather than reimplementing its internal PyTorch
API by hand. This was a deliberate choice: hand-reconstructing their model
loading / featurization code from memory carries real risk of subtle bugs
(as we saw with the KEGG compound-ID mixups in Phase 1). Driving their
actual, tested CLI is safer.

Every flag and file below was verified against the real dauparas/ProteinMPNN
GitHub source during this session (protein_mpnn_run.py argparse block,
protein_mpnn_utils.py alphabet definition, and README usage examples) --
NOT reconstructed from training-data memory. Specifically confirmed:
  - alphabet = 'ACDEFGHIKLMNPQRSTVWYX'  (21 tokens, index 0='A' ... 20='X')
  - --model_name "v_48_002" + --path_to_model_weights <folder>
    (checkpoint file resolved internally as <folder>/vanilla_model_weights/v_48_002.pt)
  - --backbone_noise 0.00, --sampling_temp "0.1"
  - --unconditional_probs_only 1 --save_probs 1
    -> single forward pass, outputs p(s_i | backbone) per position,
       written to <out_folder>/unconditional_probs_only/<name>.npz
       containing 'log_p' of shape [1, L, 21]

STILL UNVERIFIED (could not confirm without an actual repo clone + run):
  1. The exact on-disk filename ProteinMPNN gives the output .npz (assumed
     to match the PDB basename passed in -- this is standard behavior for
     their other output modes, but wasn't directly confirmed for the
     unconditional_probs_only path specifically).
  2. Whether --num_seq_per_target / --batch_size need non-default values
     to trigger unconditional_probs_only correctly (defaults of 1 used here).
  3. fixed_positions_jsonl is built via their own helper script
     (make_fixed_positions_dict.py) rather than hand-written JSON, to avoid
     a repeat of the "guessed the schema wrong" failure mode -- but the
     resulting file has not been inspected against real output.

Treat this file as a strong first draft based on verified documentation,
not as tested code. Expect to debug the subprocess calls against real
output the first time this runs, the same way we debugged Phase 1's KEGG
calls -- budget time for it.
"""

import os
import sys
import json
import shutil
import subprocess

import numpy as np

# Confirmed directly from protein_mpnn_utils.py / protein_mpnn_run.py source.
ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"

ARTIFACTS_DIR = "./pipeline_artifacts"
REPO_DIR = "./ProteinMPNN"
WORK_DIR = "./phase2_work"
MODEL_NAME = "v_48_002"  # locked decision: matches 1CHL's high-res experimental structure
BACKBONE_NOISE = "0.00"
SAMPLING_TEMP = "0.1"


class ProteinMPNNRunnerError(Exception):
    pass


class ProteinMPNNRunner:
    def __init__(self, artifacts_dir: str = ARTIFACTS_DIR, repo_dir: str = REPO_DIR,
                 work_dir: str = WORK_DIR):
        self.artifacts_dir = artifacts_dir
        self.repo_dir = repo_dir
        self.work_dir = work_dir

    def _require_repo(self):
        if not os.path.isdir(self.repo_dir):
            raise ProteinMPNNRunnerError(
                f"[ERROR] ProteinMPNN repo not found at '{self.repo_dir}'. "
                f"Clone it first:\n"
                f"  git clone --depth 1 https://github.com/dauparas/ProteinMPNN.git {self.repo_dir}"
            )
        run_script = os.path.join(self.repo_dir, "protein_mpnn_run.py")
        if not os.path.exists(run_script):
            raise ProteinMPNNRunnerError(
                f"[ERROR] '{run_script}' not found -- repo clone looks incomplete or "
                f"the directory structure has changed upstream."
            )

    def _load_manifest_and_mask(self):
        # - Plectasin (1ZFU) guard -
        # If the manifest is missing, points to the old chlorotoxin 1CHL structure,
        _EXPECTED_PDB   = "1ZFU"
        _EXPECTED_LEN   = 40
        _EXPECTED_CHAIN = "A"
        manifest_path = os.path.join(self.artifacts_dir, "phase1_manifest.json")
        mask_path     = os.path.join(self.artifacts_dir, "phase2_mask.json")

        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"[ERROR] Missing {mask_path}. Run mask_builder.py first.")

        # Load manifest if it exists; start with empty dict so checks below unify
        manifest = {}
        if os.path.exists(manifest_path):
            with open(manifest_path) as f:
                manifest = json.load(f)

        with open(mask_path) as f:
            mask = json.load(f)

        _EXPECTED_SEQ = "GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY"
        stale_pdb   = manifest.get("pdb_id", "").upper() != _EXPECTED_PDB
        file_gone   = not os.path.exists(manifest.get("raw_pdb_path", ""))
        wrong_len   = manifest.get("sequence_length", 0) != _EXPECTED_LEN
        wrong_seq   = manifest.get("sequence", "") != _EXPECTED_SEQ

        if stale_pdb or file_gone or wrong_len or wrong_seq:
            print(f"[WARN] Manifest check failed - "
                  f"pdb_id={manifest.get('pdb_id', 'missing')!r}, "
                  f"seq={manifest.get('sequence', 'missing')[:10]!r}..., "
                  f"file_on_disk={not file_gone}")
            print(f"[RECOVERY] Re-downloading {_EXPECTED_PDB} and patching manifest...")

            import urllib.request
            pdb_cache = "./pdb_cache"
            os.makedirs(pdb_cache, exist_ok=True)
            raw_path = os.path.join(pdb_cache, "pdb1zfu.pdb")
            url = f"https://files.rcsb.org/download/{_EXPECTED_PDB}.pdb"
            print(f"[RECOVERY] Fetching {url} ...")
            urllib.request.urlretrieve(url, raw_path)
            print(f"[RECOVERY] Downloaded → {raw_path}")

            manifest.update({
                "pdb_id":          _EXPECTED_PDB,
                "chain_id":        _EXPECTED_CHAIN,
                "sequence":        _EXPECTED_SEQ,
                "sequence_length": _EXPECTED_LEN,
                "raw_pdb_path":    os.path.abspath(raw_path),
                "_recovery_note":  "Auto-patched: stale/missing 1CHL manifest replaced with 1ZFU.",
            })
            os.makedirs(self.artifacts_dir, exist_ok=True)
            with open(manifest_path, "w") as f:
                json.dump(manifest, f, indent=2)
            print(f"[RECOVERY]  Manifest patched → pdb_id=1ZFU, path={raw_path}")

        # Final hard guards - should never fire after recovery above
        if "raw_pdb_path" not in manifest:
            raise ProteinMPNNRunnerError(
                "[ERROR] manifest has no 'raw_pdb_path' after recovery attempt."
            )
        if not os.path.exists(manifest["raw_pdb_path"]):
            raise FileNotFoundError(
                f"[ERROR] raw_pdb_path '{manifest['raw_pdb_path']}' still missing "
                f"after recovery - check network access to RCSB PDB."
            )

        return manifest, mask

    def _prepare_input_pdb(self, raw_pdb_path: str) -> str:
        """
        ProteinMPNN's parser expects a folder containing exactly the PDB(s)
        to design, so we copy the cached structure into an isolated inputs/
        directory rather than pointing at Phase 1's shared pdb_cache/.

        IMPORTANT: Biopython's PDBList saves files with a '.ent' extension
        (e.g. 'pdb1chl.ent'), but every documented ProteinMPNN usage example
        uses '.pdb' files. parse_multiple_chains.py likely globs for '.pdb'
        specifically, which would silently parse zero structures from a
        '.ent' file -- no crash, just an empty downstream pipeline. We copy
        with a normalized '.pdb' extension to avoid that failure mode.
        """
        inputs_dir = os.path.join(self.work_dir, "inputs")
        os.makedirs(inputs_dir, exist_ok=True)
        basename_no_ext = os.path.splitext(os.path.basename(raw_pdb_path))[0]
        dest = os.path.join(inputs_dir, f"{basename_no_ext}.pdb")
        shutil.copy(raw_pdb_path, dest)
        return inputs_dir

    def _frozen_positions_1indexed(self, mask: dict) -> list:
        """Convert mask_builder's 0-indexed frozen positions to ProteinMPNN's
        1-indexed, chain-relative position convention (confirmed via
        submit_example_4.sh: 'first amino acid in the chain corresponds to 1')."""
        frozen = []
        for key, entry in mask.items():
            if entry["status"] == "FROZEN":
                zero_idx = int(key.split("_")[1])
                frozen.append(zero_idx + 1)
        return sorted(frozen)

    def _run_subprocess(self, cmd: list, step_name: str):
        print(f"[API CALL] {step_name}: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise ProteinMPNNRunnerError(
                f"[ERROR] '{step_name}' failed (exit {result.returncode}).\n"
                f"--- stdout ---\n{result.stdout}\n"
                f"--- stderr ---\n{result.stderr}"
            )
        return result

    def run(self) -> dict:
        self._require_repo()
        manifest, mask = self._load_manifest_and_mask()

        os.makedirs(self.work_dir, exist_ok=True)
        inputs_dir = self._prepare_input_pdb(manifest["raw_pdb_path"])
        parsed_dir = os.path.join(self.work_dir, "parsed")
        os.makedirs(parsed_dir, exist_ok=True)

        chain_id = manifest["chain_id"]
        pdb_basename = os.path.splitext(os.path.basename(manifest["raw_pdb_path"]))[0]

        parsed_jsonl = os.path.join(parsed_dir, "parsed_pdbs.jsonl")
        assigned_jsonl = os.path.join(parsed_dir, "assigned_chains.jsonl")
        fixed_positions_jsonl = os.path.join(parsed_dir, "fixed_positions.jsonl")

        helper_dir = os.path.join(self.repo_dir, "helper_scripts")

        # Step 1: parse the raw PDB into ProteinMPNN's internal jsonl format
        self._run_subprocess([
            sys.executable, os.path.join(helper_dir, "parse_multiple_chains.py"),
            "--input_path", inputs_dir,
            "--output_path", parsed_jsonl,
        ], "Parse PDB into jsonl")

        # Step 2: mark our single chain as the one being designed
        self._run_subprocess([
            sys.executable, os.path.join(helper_dir, "assign_fixed_chains.py"),
            "--input_path", parsed_jsonl,
            "--output_path", assigned_jsonl,
            "--chain_list", chain_id,
        ], "Assign designed chain")

        # Step 3: build the fixed (frozen) position dictionary from our audited mask
        frozen_positions = self._frozen_positions_1indexed(mask)
        position_list_str = " ".join(str(p) for p in frozen_positions)
        self._run_subprocess([
            sys.executable, os.path.join(helper_dir, "make_fixed_positions_dict.py"),
            "--input_path", parsed_jsonl,
            "--output_path", fixed_positions_jsonl,
            "--chain_list", chain_id,
            "--position_list", position_list_str,
        ], "Build fixed-positions dict")

        # Step 4: run ProteinMPNN itself, single forward pass, unconditional
        # per-position probabilities -- exactly what Option A needs.
        out_folder = os.path.join(self.work_dir, "outputs")
        os.makedirs(out_folder, exist_ok=True)

        inference_result = self._run_subprocess([
            sys.executable, os.path.join(self.repo_dir, "protein_mpnn_run.py"),
            "--jsonl_path", parsed_jsonl,
            "--chain_id_jsonl", assigned_jsonl,
            "--fixed_positions_jsonl", fixed_positions_jsonl,
            "--out_folder", out_folder,
            "--model_name", MODEL_NAME,
            "--backbone_noise", BACKBONE_NOISE,
            "--sampling_temp", SAMPLING_TEMP,
            "--num_seq_per_target", "1",
            "--batch_size", "1",
            "--unconditional_probs_only", "1",
            "--save_probs", "1",
        ], "Run ProteinMPNN inference")
        print(f"[DEBUG] ProteinMPNN stdout:\n{inference_result.stdout}")
        print(f"[DEBUG] Full out_folder contents: {list(os.walk(out_folder))}")

        npz_path = os.path.join(out_folder, "unconditional_probs_only", f"{pdb_basename}.npz")
        if not os.path.exists(npz_path):
            candidates = []
            probs_dir = os.path.join(out_folder, "unconditional_probs_only")
            if os.path.isdir(probs_dir):
                candidates = os.listdir(probs_dir)
            raise ProteinMPNNRunnerError(
                f"[ERROR] Expected output at '{npz_path}' but it doesn't exist. "
                f"Files actually present in that folder: {candidates}. "
                f"ProteinMPNN's output naming may differ from what we assumed -- "
                f"inspect the folder and adjust npz_path construction above."
            )

        npz = np.load(npz_path)
        log_p = npz["log_p"]  # expected shape [1, L, 21] or [L, 21]
        if log_p.ndim == 3:
            log_p = log_p[0]

        probs = np.exp(log_p)  # log_p -> probabilities

        # Sanity check: rows should sum close to 1.0 if these are true softmax
        # probabilities. If not, something about our assumption of the output
        row_sums = probs.sum(axis=-1)
        if not np.allclose(row_sums, 1.0, atol=1e-2):
            print(f"[WARNING] Probability rows do not sum to ~1.0 "
                  f"(min={row_sums.min():.4f}, max={row_sums.max():.4f}). "
                  f"Verify log_p is actually log-softmax output before trusting "
                  f"the feasibility map built from this.")

        return {
            "probs": probs,          # shape [L, 21]
            "alphabet": ALPHABET,
            "sequence": manifest["sequence"],
            "npz_path": npz_path,
        }



### `Phase 2 · feasibility_map.py`

In [ ]:
"""
feasibility_map.py
--------------------
Phase 2.3: Feasibility Map Construction.

Converts ProteinMPNN's raw per-position probability matrix into the final
phase2_feasibility_map.json that Phase 4 (NSGA-II/BUG) will query directly,
without ever needing to re-invoke ProteinMPNN.

Implements Option A (locked decision): export the FULL continuous
probability distribution over the 20 standard amino acids for every
mutable position. Phase 4 applies whatever P_threshold it wants at
runtime -- this file makes no threshold decision itself.

Frozen positions are recorded with their wild-type residue only (no
distribution needed -- they're never substituted).
"""

import os
import json

import numpy as np

ARTIFACTS_DIR = "./pipeline_artifacts"

# The 20 standard amino acids, excluding ProteinMPNN's 21st token 'X' (gap /
# unknown), which is never a valid design choice for a real peptide.
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


class FeasibilityMapError(Exception):
    pass


class FeasibilityMapBuilder:
    def __init__(self, artifacts_dir: str = ARTIFACTS_DIR):
        self.artifacts_dir = artifacts_dir

    def _load_mask(self) -> dict:
        mask_path = os.path.join(self.artifacts_dir, "phase2_mask.json")
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"[ERROR] Missing {mask_path}. Run mask_builder.py first.")
        with open(mask_path) as f:
            return json.load(f)

    def build(self, probs: np.ndarray, alphabet: str, mask: dict = None) -> dict:
        """
        probs:    [L, 21] array of per-position probabilities from
                  protein_mpnn_runner.py (already exp()'d from log_p).
        alphabet: the 21-token alphabet string used to index probs
                  ('ACDEFGHIKLMNPQRSTVWYX'), so column j corresponds to
                  alphabet[j].
        mask:     the mask_builder.py output; loaded from disk if not given.
        """
        if mask is None:
            mask = self._load_mask()

        if probs.shape[0] != len(mask):
            raise FeasibilityMapError(
                f"[ERROR] Probability matrix has {probs.shape[0]} positions but "
                f"mask has {len(mask)} positions. These must match 1:1 -- "
                f"something upstream (parsing, chain selection) diverged."
            )
        if probs.shape[1] != len(alphabet):
            raise FeasibilityMapError(
                f"[ERROR] Probability matrix has {probs.shape[1]} columns but "
                f"alphabet has {len(alphabet)} tokens. Column-to-residue mapping "
                f"would be silently wrong if we proceeded."
            )

        feasibility_map = {}

        for i in range(probs.shape[0]):
            key = f"pos_{i}"
            entry = mask[key]

            if entry["status"] == "FROZEN":
                feasibility_map[key] = {
                    "status": "FROZEN",
                    "wt": entry["wt"],
                    "allowed": [entry["wt"]],
                    "layer": entry.get("layer"),
                }
                continue

            # Mutable position: build the full continuous distribution over
            # the 20 standard amino acids, dropping the 'X' gap token and
            aa_probs = {}
            for j, token in enumerate(alphabet):
                if token in STANDARD_AA:
                    aa_probs[token] = float(probs[i, j])

            total = sum(aa_probs.values())
            if total <= 0:
                raise FeasibilityMapError(
                    f"[ERROR] Position {i}: all standard-amino-acid probability "
                    f"mass is zero after dropping 'X'. Something is wrong with "
                    f"the input probability matrix at this position."
                )
            aa_probs = {aa: p / total for aa, p in aa_probs.items()}

            feasibility_map[key] = {
                "status": "MUTABLE",
                "wt": entry["wt"],
                "probabilities": dict(
                    sorted(aa_probs.items(), key=lambda kv: kv[1], reverse=True)
                ),
            }

        return feasibility_map

    def save(self, feasibility_map: dict, output_path: str = None) -> str:
        if output_path is None:
            output_path = os.path.join(self.artifacts_dir, "phase2_feasibility_map.json")
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(feasibility_map, f, indent=2)
        return output_path



### `Phase 2 · run_phase2.py    EXECUTE`

In [ ]:
# run_phase2.py
"""
Phase 2 Orchestrator.

Wires together:
  1. mask_builder.py        -- builds + validates the audited design mask
  2. protein_mpnn_runner.py -- runs ProteinMPNN once, unconditional
                                per-position probabilities
  3. feasibility_map.py     -- converts raw probabilities into the final
                                continuous-score feasibility map (Option A)

Output: ./pipeline_artifacts/phase2_mask.json
        ./pipeline_artifacts/phase2_feasibility_map.json

Both mask_builder.py and feasibility_map.py were independently tested
against synthetic/real Phase 1 output during development. protein_mpnn_runner.py
was NOT executable in the development sandbox (no network, no repo, no GPU) --
its subprocess calls are built from verified real ProteinMPNN documentation,
but this is the first genuinely untested link in the chain. Expect this
step specifically to need debugging on first real run.
"""



def execute_pipeline_phase_2():
    print("================================================================================")
    print("      PRODUCTION RUN: PHASE 2 SEQUENCE INVERSION & MASKING")
    print("================================================================================")

    # 1. Build and validate the design mask
    print("\n[STEP 1/3] Building design mask...")
    mask_builder = MaskBuilder()
    mask = mask_builder.build_mask()
    mask_path = mask_builder.save_mask(mask)
    frozen_ct = sum(1 for v in mask.values() if v["status"] == "FROZEN")
    mutable_ct = sum(1 for v in mask.values() if v["status"] == "MUTABLE")
    print(f"[SUCCESS] Mask validated: {frozen_ct} frozen / {mutable_ct} mutable. Saved to {mask_path}")

    # 2. Run ProteinMPNN inference
    print("\n[STEP 2/3] Running ProteinMPNN (single forward pass, unconditional probs)...")
    mpnn_runner = ProteinMPNNRunner()
    mpnn_result = mpnn_runner.run()
    print(f"[SUCCESS] Inference complete. Probability matrix shape: {mpnn_result['probs'].shape}")

    # 3. Build the final feasibility map
    print("\n[STEP 3/3] Building feasibility map (Option A: continuous scores)...")
    fmap_builder = FeasibilityMapBuilder()
    fmap = fmap_builder.build(mpnn_result["probs"], mpnn_result["alphabet"], mask=mask)
    fmap_path = fmap_builder.save(fmap)

    # 4. Audit summary
    print("\n================================================================================")
    print("                PHASE 2 PIPELINE AUDIT & INTEGRITY REVIEW")
    print("================================================================================")
    print(f"* Design mask:        {frozen_ct} frozen / {mutable_ct} mutable positions")
    print(f"* Feasibility map:    {fmap_path}")

    sample_mutable = next((k for k, v in fmap.items() if v["status"] == "MUTABLE"), None)
    if sample_mutable:
        top3 = list(fmap[sample_mutable]["probabilities"].items())[:3]
        print(f"* Sample [{sample_mutable}] (wt={fmap[sample_mutable]['wt']}): top candidates -> {top3}")
    print("================================================================================\n")



In [ ]:
#  RUN
execute_pipeline_phase_2()

---
## Phase 3 - COBRApy Metabolic Smoke Test
Run all 4 cells below.  Requires iML1515.xml (S4) and cobra (S1).

### `Phase 3 · fba_model_loader.py`

In [ ]:
"""
fba_model_loader.py
---------------------
Phase 3.1: iML1515 Model Loading.

Loads the genome-scale E. coli metabolic model once. Meant to be called
a single time per session (or once per Phase 4 run) -- NOT reloaded per
candidate evaluation. fba_evaluator.py uses COBRApy's `with model:`
context manager to make/revert per-candidate changes cheaply on this
one loaded instance.
"""

import os
import cobra

MODEL_PATH = "./models/iML1515.xml"

# Known published statistics for iML1515 (Monk et al. 2017), used as a
# sanity check that we loaded the right model and it parsed correctly --
EXPECTED_GENE_COUNT = 1515
EXPECTED_REACTION_COUNT = 2719
EXPECTED_METABOLITE_COUNT = 1192


class ModelLoadError(Exception):
    pass


def load_iml1515(model_path: str = MODEL_PATH) -> cobra.Model:
    if not os.path.exists(model_path):
        raise ModelLoadError(
            f"[ERROR] iML1515 model not found at '{model_path}'. Download it first:\n"
            f"  mkdir -p ./models\n"
            f"  wget -O {model_path} http://bigg.ucsd.edu/static/models/iML1515.xml"
        )

    print(f"[INFO] Loading iML1515 from '{model_path}'...")
    model = cobra.io.read_sbml_model(model_path)

    n_genes, n_reactions, n_metabolites = len(model.genes), len(model.reactions), len(model.metabolites)
    print(f"[INFO] Loaded model: {n_genes} genes, {n_reactions} reactions, {n_metabolites} metabolites.")

    # Sanity check against the model's known published dimensions. A
    # mismatch here would mean we loaded a corrupted file, a different
    if n_genes != EXPECTED_GENE_COUNT or n_reactions != EXPECTED_REACTION_COUNT or n_metabolites != EXPECTED_METABOLITE_COUNT:
        print(
            f"[WARNING] Model dimensions ({n_genes}g/{n_reactions}r/{n_metabolites}m) "
            f"don't match published iML1515 statistics "
            f"({EXPECTED_GENE_COUNT}g/{EXPECTED_REACTION_COUNT}r/{EXPECTED_METABOLITE_COUNT}m). "
            f"This could be a different model version -- proceeding, but verify this is intentional."
        )

    if model.objective is None:
        raise ModelLoadError("[ERROR] Loaded model has no objective function set.")

    baseline_solution = model.optimize()
    print(f"[INFO] Baseline (unconstrained) growth rate: {baseline_solution.objective_value:.4f} 1/h")

    if baseline_solution.status != "optimal":
        raise ModelLoadError(
            f"[ERROR] Baseline FBA did not reach an optimal solution "
            f"(status: {baseline_solution.status}). Something is wrong with the "
            f"model before we've even added a candidate demand reaction."
        )

    return model



### `Phase 3 · peptide_demand_builder.py`

In [ ]:
# peptide_demand_builder.py
# Builds the synthetic demand reaction for heterologous peptide expression.

import cobra

# One-letter amino acid code -> BiGG cytosolic metabolite ID.
# Pattern: '<name>__L_c' for L-amino acids; glycine is achiral ('gly_c').
AA_TO_BIGG_ID = {
    'A': 'ala__L_c',
    'R': 'arg__L_c',
    'N': 'asn__L_c',
    'D': 'asp__L_c',
    'C': 'cys__L_c',
    'Q': 'gln__L_c',
    'E': 'glu__L_c',
    'G': 'gly_c',
    'H': 'his__L_c',
    'I': 'ile__L_c',
    'L': 'leu__L_c',
    'K': 'lys__L_c',
    'M': 'met__L_c',
    'F': 'phe__L_c',
    'P': 'pro__L_c',
    'S': 'ser__L_c',
    'T': 'thr__L_c',
    'W': 'trp__L_c',
    'Y': 'tyr__L_c',
    'V': 'val__L_c',
}

# Translation-energy metabolite IDs in iML1515 (BiGG cytosolic).
# ATP charging: AA + ATP -> AA-tRNA + AMP + PPi  (2 high-energy bonds consumed)
TRANSLATION_METABOLITES = {
    'atp_c':  'ATP (cytosol)',
    'amp_c':  'AMP (cytosol)',
    'ppi_c':  'pyrophosphate (cytosol)',
    'gtp_c':  'GTP (cytosol)',
    'gdp_c':  'GDP (cytosol)',
    'pi_c':   'phosphate (cytosol)',
    'h2o_c':  'water (cytosol)',
    'h_c':    'proton (cytosol)',
}

DEMAND_REACTION_ID = 'DM_candidate_peptide'


class DemandBuilderError(Exception):
    pass


def validate_metabolite_ids(
    model: cobra.Model,
    residues: set,
    include_translation_energy: bool = True,
) -> None:
    # Validates that every BiGG ID this builder will use exists in the model.
    missing = []

    for residue in residues:
        if residue not in AA_TO_BIGG_ID:
            missing.append(f"'{residue}' has no entry in AA_TO_BIGG_ID at all")
            continue
        bigg_id = AA_TO_BIGG_ID[residue]
        if bigg_id not in model.metabolites:
            missing.append(f"'{residue}' -> '{bigg_id}' not found in loaded model")

    if include_translation_energy:
        for met_id, description in TRANSLATION_METABOLITES.items():
            if met_id not in model.metabolites:
                missing.append(
                    f"Translation-energy metabolite '{met_id}' ({description}) "
                    f"not found in loaded model. This is unexpected for iML1515 -- "
                    f"check that you loaded the correct model file."
                )

    if missing:
        raise DemandBuilderError(
            "[ERROR] One or more metabolite IDs could not be validated against "
            "the loaded model:\n  " + "\n  ".join(missing) +
            "\nLook up correct IDs at http://bigg.ucsd.edu/models/iML1515/metabolites"
            " -- do not guess a fix."
        )


def build_demand_reaction(
    model: cobra.Model,
    sequence: str,
    include_translation_energy: bool = True,
) -> cobra.Reaction:
    # Builds (but does not add to the model) a demand reaction representing
    # heterologous expression of `sequence` in E. coli.

    if not sequence:
        raise DemandBuilderError("[ERROR] Cannot build a demand reaction for an empty sequence.")

    n_residues = len(sequence)

    residue_counts = {}
    for residue in sequence:
        residue_counts[residue] = residue_counts.get(residue, 0) + 1

    validate_metabolite_ids(model, set(residue_counts.keys()), include_translation_energy)

    reaction = cobra.Reaction(DEMAND_REACTION_ID)
    reaction.name = (
        "Candidate peptide synthesis demand (AA precursors + translation energy)"
        if include_translation_energy
        else "Candidate peptide synthesis demand (amino-acid precursors only)"
    )
    reaction.lower_bound = 0
    reaction.upper_bound = 1000

    coefficients = {}

    # -- Amino acid precursor drain (identical to legacy behavior) ----------------
    for residue, count in residue_counts.items():
        met = model.metabolites.get_by_id(AA_TO_BIGG_ID[residue])
        coefficients[met] = -float(count)

    if include_translation_energy:
        # -- Aminoacyl-tRNA charging ----------------------------------------------
        # Each residue requires: ATP + H2O -> AMP + PPi + H
        _add(coefficients, model, 'atp_c',  -float(n_residues))   # consumed
        _add(coefficients, model, 'h2o_c',  -float(n_residues))   # consumed
        _add(coefficients, model, 'amp_c',  +float(n_residues))   # produced
        _add(coefficients, model, 'ppi_c',  +float(n_residues))   # produced
        _add(coefficients, model, 'h_c',    +float(n_residues))   # produced

        # -- Ribosomal elongation -------------------------------------------------
        # Each peptide bond requires: 2 GTP + 2 H2O -> 2 GDP + 2 Pi + 2 H
        n_bonds = max(n_residues - 1, 0)
        _add(coefficients, model, 'gtp_c',  -2.0 * n_bonds)   # consumed
        _add(coefficients, model, 'h2o_c',  -2.0 * n_bonds)   # consumed (additional)
        _add(coefficients, model, 'gdp_c',  +2.0 * n_bonds)   # produced
        _add(coefficients, model, 'pi_c',   +2.0 * n_bonds)   # produced
        _add(coefficients, model, 'h_c',    +2.0 * n_bonds)   # produced (additional)

    reaction.add_metabolites(coefficients)
    return reaction


def _add(coefficients: dict, model: cobra.Model, met_id: str, delta: float) -> None:
    # Accumulates stoichiometry for a metabolite, handling multiple additions
    # to the same metabolite (e.g. h2o_c appears in both charging and elongation).
    met = model.metabolites.get_by_id(met_id)
    coefficients[met] = coefficients.get(met, 0.0) + delta


### `Phase 3 · fba_evaluator.py`

In [ ]:
# fba_evaluator.py
# Runs FBA to score a candidate sequence's metabolic production burden (MPB).

# build_demand_reaction and DEMAND_REACTION_ID defined in the cell above


class FBAEvaluatorError(Exception):
    pass


# Peptide-equivalent flux forced through the demand reaction per evaluation.
PEPTIDE_DEMAND_FLUX = 0.01

# Set to False only to reproduce pre-PR results for direct comparison.
# True is the default and the scientifically correct setting.
INCLUDE_TRANSLATION_ENERGY = True


def evaluate_candidate(
    model,
    sequence: str,
    baseline_growth: float = None,
    include_translation_energy: bool = INCLUDE_TRANSLATION_ENERGY,
) -> dict:
    # Scores a single candidate sequence's Metabolic Production Burden (MPB).
    #

    if baseline_growth is None:
        baseline_solution = model.optimize()
        if baseline_solution.status != 'optimal':
            raise FBAEvaluatorError(
                f"[ERROR] Baseline FBA did not reach an optimal solution "
                f"(status: {baseline_solution.status}). Cannot compute MPB."
            )
        baseline_growth = baseline_solution.objective_value

    with model:
        demand_reaction = build_demand_reaction(
            model,
            sequence,
            include_translation_energy=include_translation_energy,
        )
        model.add_reactions([demand_reaction])
        model.reactions.get_by_id(DEMAND_REACTION_ID).lower_bound = PEPTIDE_DEMAND_FLUX

        candidate_solution = model.optimize()

        if candidate_solution.status != 'optimal':
            print(
                f"[WARNING] Candidate infeasible at PEPTIDE_DEMAND_FLUX={PEPTIDE_DEMAND_FLUX} "
                f"(status: {candidate_solution.status}). Scoring as 100% growth reduction. "
                f"If this fires for most candidates, PEPTIDE_DEMAND_FLUX is too high."
            )
            candidate_growth = 0.0
        else:
            candidate_growth = candidate_solution.objective_value

    # model automatically reverts here (COBRApy context manager behavior)

    growth_delta = baseline_growth - candidate_growth
    growth_percent_reduction = (
        100.0 * (1.0 - candidate_growth / baseline_growth)
        if baseline_growth > 0 else 100.0
    )

    return {
        'sequence':                 sequence,
        'baseline_growth':          baseline_growth,
        'candidate_growth':         candidate_growth,
        'growth_delta':             growth_delta,
        'growth_percent_reduction': growth_percent_reduction,
        'translation_energy':       include_translation_energy,
    }


### `Phase 3 · run_phase3.py    EXECUTE`

In [ ]:
# run_phase3.py
"""
Phase 3 Orchestrator / Smoke Test.

Unlike run_phase1.py and run_phase2.py, this is NOT a "real" production
run -- Phase 3's actual deliverable is the reusable fba_evaluator.py
function that Phase 4 will call per-candidate. Since Phase 4 doesn't
exist yet, there are no real evolved candidates to score.

This script instead proves the mechanism works end-to-end:
  1. Load iML1515 once.
  2. Evaluate the wild-type 1ZFU (Plectasin) sequence itself (sanity baseline --
     the unmodified natural peptide should get *some* reasonable,
     non-zero, non-infeasible burden score).
  3. Evaluate 1-2 synthetically mutated sequences (single substitutions
     at mutable positions from Phase 2's mask) to confirm the score
     actually changes with composition, in a plausible direction.
"""

import json


ARTIFACTS_DIR = "./pipeline_artifacts"


def load_wild_type_sequence() -> str:
    with open(f"{ARTIFACTS_DIR}/phase1_manifest.json") as f:
        manifest = json.load(f)
    return manifest["sequence"]


def make_test_mutant(wild_type: str) -> str:
    """
    Builds one synthetic mutant for smoke-testing only: swaps the first
    mutable (non-frozen) position to Tryptophan (W) -- the most
    metabolically expensive residue (highest ATP cost in Phase 1's
    table), so we should see the burden score visibly increase versus
    wild-type if the mechanism is working correctly.
    """
    with open(f"{ARTIFACTS_DIR}/phase2_mask.json") as f:
        mask = json.load(f)

    for i in range(len(wild_type)):
        entry = mask[f"pos_{i}"]
        if entry["status"] == "MUTABLE" and entry["wt"] != "W":
            mutant = wild_type[:i] + "W" + wild_type[i + 1:]
            return mutant, i
    raise RuntimeError("No suitable mutable position found for smoke test.")


def execute_pipeline_phase_3():
    print("================================================================================")
    print("      PHASE 3 SMOKE TEST: METABOLIC CONSTRAINT FEEDBACK (COBRApy / iML1515)")
    print("================================================================================")

    print("\n[STEP 1/3] Loading iML1515...")
    model = load_iml1515()

    print("\n[STEP 2/3] Evaluating wild-type 1ZFU (Plectasin) sequence...")
    wild_type = load_wild_type_sequence()
    wt_result = evaluate_candidate(model, wild_type)
    print(f"[SUCCESS] Wild-type MPB score: {wt_result['growth_percent_reduction']:.4f}% growth reduction")

    print("\n[STEP 3/3] Evaluating a synthetic high-cost mutant (smoke test)...")
    mutant_seq, mutated_pos = make_test_mutant(wild_type)
    mutant_result = evaluate_candidate(model, mutant_seq, baseline_growth=wt_result["baseline_growth"])
    print(f"[SUCCESS] Mutant (pos_{mutated_pos} -> W) MPB score: {mutant_result['growth_percent_reduction']:.4f}% growth reduction")

    print("\n================================================================================")
    print("                PHASE 3 SMOKE TEST AUDIT")
    print("================================================================================")
    print(f"* Baseline (unconstrained) growth: {wt_result['baseline_growth']:.4f} 1/h")
    print(f"* Wild-type candidate growth:      {wt_result['candidate_growth']:.4f} 1/h  ({wt_result['growth_percent_reduction']:.4f}% reduction)")
    print(f"* Mutant candidate growth:         {mutant_result['candidate_growth']:.4f} 1/h  ({mutant_result['growth_percent_reduction']:.4f}% reduction)")

    if mutant_result["growth_percent_reduction"] >= wt_result["growth_percent_reduction"]:
        print(f"* Sanity check PASSED: substituting a cheap residue for Trp increased (or held) the burden score, as expected.")
    else:
        print(f"* [WARNING] Sanity check FAILED: Trp substitution decreased the burden score. "
              f"This is suspicious -- Trp should be among the most metabolically expensive residues. "
              f"Investigate before trusting fba_evaluator.py's output.")
    print("================================================================================\n")



In [ ]:
#  RUN
execute_pipeline_phase_3()

---
## Phase 4 - NSGA-II + MCTS Evolutionary Loop
Run all 5 cells below. Expect ~15-20 min for 50 generations.

### `Phase 4 · sequence_utils.py`

In [ ]:
"""
sequence_utils.py -- Phase 4 shared stateless utilities.
All other Phase 4 files import from here.

ADDED: compute_population_entropy() for adaptive stagnation detection.
"""
import json, math, random
from collections import Counter

FEASIBILITY_MAP_PATH = "./pipeline_artifacts/phase2_feasibility_map.json"
MASK_PATH            = "./pipeline_artifacts/phase2_mask.json"
WILD_TYPE            = "GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY"

def load_feasibility_map(path=FEASIBILITY_MAP_PATH):
    with open(path) as f: return json.load(f)

def load_mask(path=MASK_PATH):
    with open(path) as f: return json.load(f)

def get_mutable_positions(mask):
    return sorted([int(k.split("_")[1]) for k,v in mask.items() if v["status"]=="MUTABLE"])

def compute_structural_score(sequence, feasibility_map):
    """Sum of log P(aa | backbone) at mutable positions. Higher = better."""
    score = 0.0
    for i, aa in enumerate(sequence):
        entry = feasibility_map.get(f"pos_{i}")
        if entry and entry["status"] == "MUTABLE":
            score += math.log(max(entry["probabilities"].get(aa, 1e-10), 1e-10))
    return score

def sample_sequence(feasibility_map, mask, wild_type=WILD_TYPE):
    seq = list(wild_type)
    for i in range(len(wild_type)):
        entry = feasibility_map.get(f"pos_{i}")
        if entry and entry["status"] == "MUTABLE":
            probs = entry["probabilities"]
            aas, weights = list(probs.keys()), list(probs.values())
            seq[i] = random.choices(aas, weights=weights, k=1)[0]
    return "".join(seq)

def apply_mutation(sequence, position, new_residue):
    seq = list(sequence); seq[position] = new_residue; return "".join(seq)

def validate_sequence(sequence, mask, wild_type=WILD_TYPE):
    violations = [
        f"pos_{i}: frozen {wt} -> {aa}"
        for i,(aa,wt) in enumerate(zip(sequence, wild_type))
        if mask[f"pos_{i}"]["status"]=="FROZEN" and aa!=wt
    ]
    return len(violations)==0, violations

def dominates(a, b):
    not_worse = (a["structural_score"] >= b["structural_score"] and
                 a["mpb_score"] <= b["mpb_score"])
    strictly_better = (a["structural_score"] > b["structural_score"] or
                       a["mpb_score"] < b["mpb_score"])
    return not_worse and strictly_better

def compute_population_entropy(population, mutable_positions):
    """
    Shannon entropy of amino acid distribution at each mutable position
    across the current population. Returns mean entropy in bits across
    all mutable positions.

    Maximum possible entropy = log2(20) ≈ 4.32 bits (all 20 AAs equally
    likely at every position). Low entropy = homogeneous population.

    Used by NSGA2Engine to detect stagnation: if entropy falls below a
    threshold AND fitness has plateaued, the adaptive boost fires.
    """
    if not population or not mutable_positions:
        return 0.0

    entropies = []
    for pos in mutable_positions:
        counts = Counter(c["sequence"][pos] for c in population)
        total  = sum(counts.values())
        h = -sum((n/total) * math.log2(n/total) for n in counts.values() if n > 0)
        entropies.append(h)

    return sum(entropies) / len(entropies)


### `Phase 4 · mcts_operator.py`

In [ ]:
"""
mcts_operator.py -- Phase 4 MCTS-guided mutation operator.

DEPTH-2 VIA SEQUENTIAL MCTS
=============================
The naive depth-2 tree approach failed because with ~380 possible
single-residue mutations at each level, 50 rollouts never exhausted
enough of the root's untried children to descend deeper -- every
rollout spent itself expanding depth-1 nodes and zero depth-2 nodes
were ever reached. Confirmed by test: 10/10 mutations were depth-1.

Fix: run depth-1 MCTS twice sequentially:
  Step 1: run MCTS on parent_sequence with n_rollouts * STEP1_FRACTION
          budget → returns best single-mutation mid-sequence.
  Step 2: run MCTS on mid-sequence with remaining budget, but exclude
          the position already mutated in step 1 (enforces that the
          two mutations are always at different positions).

This is mathematically equivalent to depth-2 tree search under the
same feasibility-map prior, without the branching-factor tractability
problem. Guaranteed to produce a 2-mutation offspring every call.
"""
import math, random

UCB1_C       = math.sqrt(2)
STEP1_FRACTION = 0.6   # fraction of rollout budget used for first mutation


class _Node:
    __slots__ = ["sequence","mutation","parent","children","visits","total_reward","_untried"]
    def __init__(self, sequence, mutation=None, parent=None):
        self.sequence     = sequence
        self.mutation     = mutation
        self.parent       = parent
        self.children     = []
        self.visits       = 0
        self.total_reward = 0.0
        self._untried     = None

    @property
    def mean_reward(self):
        return self.total_reward / self.visits if self.visits > 0 else 0.0

    def ucb1(self):
        if self.visits == 0: return float("inf")
        if not self.parent or self.parent.visits == 0: return self.mean_reward
        return (self.mean_reward +
                UCB1_C * math.sqrt(math.log(self.parent.visits) / self.visits))

    def is_fully_expanded(self): return self._untried is not None and len(self._untried) == 0
    def best_ucb_child(self):    return max(self.children, key=lambda c: c.ucb1())
    def best_mean_child(self):   return max(self.children, key=lambda c: c.mean_reward) if self.children else None


class MCTSOperator:
    def __init__(self, feasibility_map, mask, n_rollouts=50):
        self.feasibility_map = feasibility_map
        self.mask            = mask
        self.n_rollouts      = n_rollouts
        self._mutable        = [int(k.split("_")[1])
                                 for k,v in mask.items() if v["status"]=="MUTABLE"]

    def _build_untried(self, sequence, exclude_positions=None):
        """Weighted-shuffle of (pos, residue) candidates, optionally excluding positions."""
        exclude_positions = exclude_positions or set()
        candidates = []
        for pos in self._mutable:
            if pos in exclude_positions: continue
            entry = self.feasibility_map[f"pos_{pos}"]
            for aa, prob in entry["probabilities"].items():
                if aa != sequence[pos]:
                    candidates.append((pos, aa, prob))
        if not candidates: return []
        total = sum(c[2] for c in candidates)
        candidates.sort(
            key=lambda c: -math.log(max(random.random(), 1e-10)) / (c[2] / total)
        )
        return [(pos, aa) for pos, aa, _ in candidates]

    def _run_mcts(self, sequence, n_rollouts, exclude_positions=None):
        """Single-level MCTS returning the best mutated sequence found."""
        root = _Node(sequence); root.visits = 1

        for _ in range(n_rollouts):
            node = root
            # Selection
            while node.is_fully_expanded() and node.children:
                node = node.best_ucb_child()
            # Expansion
            if not node.is_fully_expanded():
                if node._untried is None:
                    node._untried = self._build_untried(node.sequence, exclude_positions)
                if node._untried:
                    pos, aa = node._untried.pop(0)
                    child = _Node(apply_mutation(node.sequence, pos, aa), (pos,aa), node)
                    node.children.append(child); node = child
            # Rollout (instant structural score)
            reward = compute_structural_score(node.sequence, self.feasibility_map)
            # Backprop
            cur = node
            while cur:
                cur.visits += 1; cur.total_reward += reward; cur = cur.parent

        best = root.best_mean_child()
        return best.sequence if best else sequence, best.mutation if best else None

    def mutate(self, parent_sequence):
        """
        Two sequential depth-1 MCTS calls producing a 2-mutation offspring.
        Step 1 finds the best single mutation; Step 2 finds the best second
        mutation from that result, constrained to a different position.
        """
        n1 = max(1, int(self.n_rollouts * STEP1_FRACTION))
        n2 = max(1, self.n_rollouts - n1)

        # Step 1: best single mutation
        mid_seq, first_mutation = self._run_mcts(parent_sequence, n1)

        # Step 2: best second mutation, excluding the position already changed
        already_mutated = set()
        if first_mutation: already_mutated.add(first_mutation[0])

        result_seq, _ = self._run_mcts(mid_seq, n2, exclude_positions=already_mutated)
        return result_seq


### `Phase 4 · nsga2_engine.py`

In [ ]:
"""
nsga2_engine.py -- NSGA-II core + generational loop.

ADDED: Adaptive stagnation recovery via Shannon entropy monitoring.

HOW IT WORKS
============
Each generation, after survivor selection:
  1. Compute population-level Shannon entropy (bits) across all mutable
     positions. High entropy = diverse population. Low = homogeneous.
  2. Check if best fitness (struct + mpb) has improved within the last
     STAGNATION_WINDOW generations. If not → stagnated.
  3. If BOTH entropy < ENTROPY_THRESHOLD AND stagnated → fire boost:
       a. Inject BOOST_INJECT_COUNT fresh random sequences (replaces
          lowest-ranked survivors) to immediately widen the gene pool.
       b. Temporarily raise MCTS rollouts to BOOST_ROLLOUTS for
          BOOST_DURATION generations, encouraging wider mutation search.
       c. Start BOOST_COOLDOWN countdown. Boost cannot re-fire until
          cooldown expires, preventing oscillation during real convergence.
  4. Cooldown check uses separate condition from stagnation check so
     that legitimate convergence (entropy dropping + fitness still
     improving) never triggers the boost.
"""
import os, random, math

# - Stagnation / boost parameters -
STAGNATION_WINDOW  = 5     # generations without improvement to declare stagnation
ENTROPY_THRESHOLD  = 1.5   # bits; max possible = log2(20) ≈ 4.32; below this = low diversity
BOOST_ROLLOUTS     = 150   # MCTS rollouts during boost (3x the default 50)
BOOST_DURATION     = 5     # generations to run boosted MCTS after trigger
BOOST_INJECT_COUNT = 5     # random sequences injected into population on trigger
BOOST_COOLDOWN     = 10    # generations before boost can re-fire


# - NSGA-II primitives -

def fast_non_dominated_sort(population):
    n = len(population)
    dom_count = [0]*n
    dom_set   = [[] for _ in range(n)]
    fronts    = [[]]
    for i in range(n):
        for j in range(n):
            if i==j: continue
            if dominates(population[i], population[j]):   dom_set[i].append(j)
            elif dominates(population[j], population[i]): dom_count[i] += 1
        if dom_count[i] == 0:
            population[i]["rank"] = 0; fronts[0].append(i)
    k = 0
    while fronts[k]:
        nxt = []
        for i in fronts[k]:
            for j in dom_set[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0:
                    population[j]["rank"] = k+1; nxt.append(j)
        k += 1; fronts.append(nxt)
    return [f for f in fronts if f]


def crowding_distance(front_indices, population):
    n = len(front_indices)
    for idx in front_indices: population[idx]["crowding_distance"] = 0.0
    if n <= 2:
        for idx in front_indices: population[idx]["crowding_distance"] = float("inf")
        return
    for obj in ["structural_score", "mpb_score"]:
        srt = sorted(front_indices, key=lambda i: population[i][obj])
        population[srt[0]]["crowding_distance"]  = float("inf")
        population[srt[-1]]["crowding_distance"] = float("inf")
        rng = population[srt[-1]][obj] - population[srt[0]][obj]
        if rng == 0: continue
        for i in range(1, n-1):
            population[srt[i]]["crowding_distance"] += (
                population[srt[i+1]][obj] - population[srt[i-1]][obj]) / rng


def tournament_select(population):
    a, b = random.sample(population, 2)
    ra, rb = a.get("rank",float("inf")), b.get("rank",float("inf"))
    if ra < rb: return a
    if rb < ra: return b
    return a if a.get("crowding_distance",0) >= b.get("crowding_distance",0) else b


def select_survivors(combined, target_size):
    fronts = fast_non_dominated_sort(combined)
    for f in fronts: crowding_distance(f, combined)
    survivors = []
    for f in fronts:
        if len(survivors) + len(f) <= target_size:
            survivors.extend([combined[i] for i in f])
        else:
            rem = target_size - len(survivors)
            srt = sorted(f, key=lambda i: combined[i].get("crowding_distance",0), reverse=True)
            survivors.extend([combined[i] for i in srt[:rem]]); break
    return survivors


# - Engine -

class NSGA2Engine:
    def __init__(self, feasibility_map, mask, model, baseline_growth,
                 population_size=30, n_generations=50, n_mcts_rollouts=50,
                 checkpoint_dir="./phase4_checkpoints", use_mcts=True):

        self.feasibility_map  = feasibility_map
        self.mask             = mask
        self.model            = model
        self.baseline_growth  = baseline_growth
        self.population_size  = population_size
        self.n_generations    = n_generations
        self.base_rollouts    = n_mcts_rollouts
        self.checkpoint_dir   = checkpoint_dir
        self.use_mcts         = use_mcts
        self.mutable_positions= get_mutable_positions(mask)

        self.mcts        = MCTSOperator(feasibility_map, mask, n_mcts_rollouts)
        self._fba_cache  = {}
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Stagnation / boost state
        self._fitness_history  = []   # (best_struct, best_mpb) per generation
        self._boost_remaining  = 0    # generations left in current boost
        self._cooldown_remaining = 0  # generations until boost can re-fire
        self._boost_count      = 0    # total times boost has fired (for audit)

    # - Evaluation -

    def _mpb(self, sequence):
        if sequence not in self._fba_cache:
            r = evaluate_candidate(self.model, sequence, self.baseline_growth)
            self._fba_cache[sequence] = r["growth_percent_reduction"]
        return self._fba_cache[sequence]

    def _make_candidate(self, sequence, generation):
        return {
            "sequence":          sequence,
            "structural_score":  compute_structural_score(sequence, self.feasibility_map),
            "mpb_score":         self._mpb(sequence),
            "generation":        generation,
            "rank":              None,
            "crowding_distance": 0.0,
        }

    def _eval_pop(self, sequences, generation):
        return [self._make_candidate(s, generation) for s in sequences]

    def _init_population(self):
        seqs = {WILD_TYPE}
        while len(seqs) < self.population_size:
            seqs.add(sample_sequence(self.feasibility_map, self.mask))
        return list(seqs)

    # - Mutation -

    def _mutate(self, parent_seq):
        if self.use_mcts:
            return self.mcts.mutate(parent_seq)
        pos     = random.choice(self.mutable_positions)
        entry   = self.feasibility_map[f"pos_{pos}"]
        choices = [aa for aa in entry["probabilities"] if aa != parent_seq[pos]]
        return apply_mutation(parent_seq, pos, random.choice(choices)) if choices else parent_seq

    # - Stagnation detection & adaptive boost -

    def _check_stagnated(self):
        """
        Returns True if neither best_struct nor best_mpb has improved
        beyond epsilon in the last STAGNATION_WINDOW generations.
        """
        if len(self._fitness_history) < STAGNATION_WINDOW:
            return False
        window   = self._fitness_history[-STAGNATION_WINDOW:]
        best_struct_old, best_mpb_old = window[0]
        best_struct_now, best_mpb_now = window[-1]
        struct_improved = (best_struct_now - best_struct_old) > 0.01
        mpb_improved    = (best_mpb_old - best_mpb_now)       > 0.001
        return not struct_improved and not mpb_improved

    def _maybe_fire_boost(self, population, generation):
        """
        Fires adaptive diversity boost when entropy < threshold AND
        stagnated AND cooldown has expired. Returns (population, boosted: bool).
        """
        if self._cooldown_remaining > 0:
            return population, False

        entropy   = compute_population_entropy(population, self.mutable_positions)
        stagnated = self._check_stagnated()

        if entropy >= ENTROPY_THRESHOLD or not stagnated:
            return population, False

        # - FIRE BOOST -
        self._boost_remaining    = BOOST_DURATION
        self._cooldown_remaining = BOOST_COOLDOWN
        self._boost_count       += 1
        self.mcts.n_rollouts     = BOOST_ROLLOUTS

        # Inject fresh random sequences, replacing lowest-ranked survivors
        injected_seqs  = [sample_sequence(self.feasibility_map, self.mask)
                          for _ in range(BOOST_INJECT_COUNT)]
        injected       = self._eval_pop(injected_seqs, generation)
        sorted_pop     = sorted(population,
                                key=lambda c: (c.get("rank", 99),
                                               -c.get("crowding_distance", 0)))
        combined = sorted_pop[:-BOOST_INJECT_COUNT] + injected
        return combined, True

    def _update_boost_state(self):
        """Called once per generation to tick down boost and cooldown counters."""
        if self._boost_remaining > 0:
            self._boost_remaining -= 1
            if self._boost_remaining == 0:
                self.mcts.n_rollouts = self.base_rollouts   # restore normal rollouts
        if self._cooldown_remaining > 0:
            self._cooldown_remaining -= 1

    # - Main loop -

    def run(self, pareto_archive=None):
        if pareto_archive is None: pareto_archive = ParetoArchive()

        mode = "MCTS" if self.use_mcts else "RANDOM"
        print(f"\n{'='*72}")
        print(f"  NSGA-II [{mode}] | pop={self.population_size} | gen={self.n_generations}")
        print(f"  Entropy threshold={ENTROPY_THRESHOLD} bits | "
              f"Stagnation window={STAGNATION_WINDOW} gen | "
              f"Boost rollouts={BOOST_ROLLOUTS} for {BOOST_DURATION} gen")
        print(f"{'='*72}")

        population = self._eval_pop(self._init_population(), 0)
        pareto_archive.update(population)
        history = []

        for gen in range(1, self.n_generations + 1):

            # Sort current population
            fronts = fast_non_dominated_sort(population)
            for f in fronts: crowding_distance(f, population)

            # Generate offspring
            offspring_seqs = [
                self._mutate(tournament_select(population)["sequence"])
                for _ in range(self.population_size)
            ]
            offspring  = self._eval_pop(offspring_seqs, gen)
            population = select_survivors(population + offspring, self.population_size)
            pareto_archive.update(population)

            # Compute metrics
            best_struct = max(c["structural_score"] for c in population)
            best_mpb    = min(c["mpb_score"]        for c in population)
            entropy     = compute_population_entropy(population, self.mutable_positions)
            self._fitness_history.append((best_struct, best_mpb))

            # Adaptive boost check
            population, boosted = self._maybe_fire_boost(population, gen)
            boost_tag = f" [BOOST #{self._boost_count} FIRED]" if boosted else (
                        f" [boost {self._boost_remaining}gen left]" if self._boost_remaining > 0 else "")

            self._update_boost_state()

            history.append({
                "generation":        gen,
                "pareto_front_size": len(pareto_archive.front),
                "cache_size":        len(self._fba_cache),
                "best_structural":   best_struct,
                "best_mpb":          best_mpb,
                "entropy_bits":      round(entropy, 4),
                "boost_fired":       boosted,
                "total_boosts":      self._boost_count,
            })

            print(f"  gen {gen:03d} | pareto={len(pareto_archive.front):3d} | "
                  f"cache={len(self._fba_cache):4d} | "
                  f"struct={best_struct:.3f} | mpb={best_mpb:.4f}% | "
                  f"H={entropy:.2f}b{boost_tag}")

            if gen % 10 == 0:
                ckpt = os.path.join(self.checkpoint_dir, f"checkpoint_gen_{gen:03d}.json")
                pareto_archive.checkpoint(ckpt, {"generation": gen, "history": history})
                print(f"  [CHECKPOINT] {ckpt}")

        print(f"\n  Done. Pareto={len(pareto_archive.front)} | "
              f"FBA calls={len(self._fba_cache)} | "
              f"Total boosts fired={self._boost_count}")
        return pareto_archive, history


### `Phase 4 · pareto_archive.py`

In [ ]:
"""
pareto_archive.py -- Tracks the running Pareto frontier across generations.
Handles deduplication, checkpointing, and final output.
"""
import os, json
from datetime import datetime, timezone


class ParetoArchive:
    def __init__(self):
        self.front = []          # current non-dominated set
        self.all_evaluated = []  # every candidate ever seen (for post-analysis)

    def update(self, population):
        """Merge new candidates into the Pareto front, deduplicating by sequence."""
        self.all_evaluated.extend(population)
        combined = self.front + population

        new_front = []
        for i, cand in enumerate(combined):
            if not any(dominates(combined[j], cand) for j in range(len(combined)) if j!=i):
                new_front.append(cand)

        # Deduplicate by sequence, keeping latest version of each
        seen, deduped = set(), []
        for c in reversed(new_front):
            if c["sequence"] not in seen:
                seen.add(c["sequence"]); deduped.append(c)
        self.front = deduped

    def checkpoint(self, path, metadata=None):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w") as f:
            json.dump({
                "timestamp":    datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "pareto_front": self.front,
                "metadata":     metadata or {},
            }, f, indent=2)

    def save_final(self, path="./pipeline_artifacts/phase4_pareto_front.json"):
        self.checkpoint(path, {"type": "final_pareto_front",
                                "n_candidates": len(self.front)})
        return path

    @classmethod
    def load_checkpoint(cls, path):
        archive = cls()
        with open(path) as f: data = json.load(f)
        archive.front = data["pareto_front"]
        return archive


### `Phase 4 · run_phase4.py    EXECUTE`

In [ ]:
"""
run_phase4.py -- Phase 4 orchestrator.

Loads Phase 1-3 outputs, runs the NSGA-II + MCTS evolutionary loop,
checkpoints every 10 generations, saves the final Pareto front.

Setup required before running:
  pip install cobra pymoo matplotlib pandas
  (Phase 1-3 already completed and ./pipeline_artifacts/ populated)
"""
import json

# - Configuration -
POPULATION_SIZE  = 30
N_GENERATIONS    = 50
N_MCTS_ROLLOUTS  = 50
CHECKPOINT_DIR   = "./phase4_checkpoints"
FINAL_OUTPUT     = "./pipeline_artifacts/phase4_pareto_front.json"


def execute_pipeline_phase_4():
    print("="*72)
    print("  PHASE 4: PARETO EVOLUTIONARY LOOP")
    print("  Feasibility-Conditioned Multi-Objective GA with MCTS Mutations")
    print("="*72)

    # Load Phase 2 artifacts
    print("\n[1/4] Loading Phase 2 feasibility map and mask...")
    feasibility_map = load_feasibility_map()
    mask            = load_mask()
    print(f"      Feasibility map: {sum(1 for v in feasibility_map.values() if v['status']=='MUTABLE')} mutable positions")

    # Load Phase 3 metabolic model
    print("\n[2/4] Loading iML1515 and computing baseline growth rate...")
    model = load_iml1515()
    baseline_result  = evaluate_candidate(model, WILD_TYPE)
    baseline_growth  = baseline_result["baseline_growth"]
    wt_mpb           = baseline_result["growth_percent_reduction"]
    print(f"      Baseline growth: {baseline_growth:.4f} 1/h")
    print(f"      Wild-type MPB:   {wt_mpb:.4f}%")

    # Run NSGA-II + MCTS
    print("\n[3/4] Running evolutionary optimization...")
    engine = NSGA2Engine(
        feasibility_map  = feasibility_map,
        mask             = mask,
        model            = model,
        baseline_growth  = baseline_growth,
        population_size  = POPULATION_SIZE,
        n_generations    = N_GENERATIONS,
        n_mcts_rollouts  = N_MCTS_ROLLOUTS,
        checkpoint_dir   = CHECKPOINT_DIR,
        use_mcts         = True,
    )
    pareto_archive, history = engine.run()

    # Save final output
    print("\n[4/4] Saving final Pareto front...")
    path = pareto_archive.save_final(FINAL_OUTPUT)

    # Audit
    front = pareto_archive.front
    print(f"\n{'='*72}")
    print(f"  PHASE 4 COMPLETE")
    print(f"{'='*72}")
    print(f"  Pareto front candidates: {len(front)}")
    print(f"  Wild-type reference:     struct={baseline_result.get('structural_score','N/A')} | mpb={wt_mpb:.4f}%")
    if front:
        best_struct = max(front, key=lambda c: c["structural_score"])
        best_mpb    = min(front, key=lambda c: c["mpb_score"])
        print(f"  Best structural score:   {best_struct['structural_score']:.4f} -> {best_struct['sequence']}")
        print(f"  Best MPB (lowest cost):  {best_mpb['mpb_score']:.4f}% -> {best_mpb['sequence']}")
    print(f"  Output: {path}")
    print(f"{'='*72}\n")

    return pareto_archive, history



In [ ]:
#  RUN
execute_pipeline_phase_4()

---
## Phase 4 Enhanced - Graph-NSGA-II + Learned Perturbation
Run all 4 cells below for the enhanced optimizer. Produces a separate Pareto front for comparison.

### `Phase 4E · fitness_graph.py`

In [ ]:
"""
fitness_graph.py -- Fitness landscape graph for local minima detection.

Builds a graph where every evaluated sequence is a node, and edges
connect sequences within Hamming distance <= 2 at mutable positions.
Used to detect when the search is trapped in a local optimum and to
identify which regions of sequence space are underexplored.

LOCAL MINIMA DEFINITION (multi-objective):
A sequence is a local optimum if none of its graph neighbors dominate it.
If the entire reachable neighborhood contains no dominating solution,
the search has no gradient to follow via single/double mutations --
this is the signal to fire a learned perturbation.
"""
import sys, os


def hamming_distance_mutable(seq_a, seq_b, mutable_positions):
    """Hamming distance counting only mutable positions."""
    return sum(1 for pos in mutable_positions if seq_a[pos] != seq_b[pos])


class FitnessGraph:
    def __init__(self, mutable_positions, max_edge_distance=2):
        self.mutable_positions  = mutable_positions
        self.max_edge_distance  = max_edge_distance
        self._nodes             = {}   # sequence -> candidate dict
        self._adjacency         = {}   # sequence -> set of neighbor sequences

    def add_candidates(self, candidates):
        """Add a batch of evaluated candidates, building edges lazily."""
        new_seqs = []
        for cand in candidates:
            seq = cand["sequence"]
            if seq not in self._nodes:
                self._nodes[seq]      = cand
                self._adjacency[seq]  = set()
                new_seqs.append(seq)

        # Connect new sequences to all existing nodes within max_edge_distance
        existing = list(self._nodes.keys())
        for new_seq in new_seqs:
            for existing_seq in existing:
                if new_seq == existing_seq: continue
                d = hamming_distance_mutable(new_seq, existing_seq, self.mutable_positions)
                if d <= self.max_edge_distance:
                    self._adjacency[new_seq].add(existing_seq)
                    self._adjacency[existing_seq].add(new_seq)

    def get_neighbors(self, sequence):
        """Return all evaluated sequences within max_edge_distance mutations."""
        return [self._nodes[s] for s in self._adjacency.get(sequence, set())]

    def is_local_optimum(self, candidate):
        """
        Returns True if no neighbor in the graph dominates this candidate.
        In a local optimum, single/double mutations have no improving direction.
        """
        neighbors = self.get_neighbors(candidate["sequence"])
        if not neighbors:
            return False  # isolated node -- not enough info to declare local opt
        return not any(dominates(n, candidate) for n in neighbors)

    def get_underexplored_positions(self, population, top_k=5):
        """
        Returns the mutable positions least represented in the population's
        current sequence diversity. Used to direct perturbations toward
        regions where little variation has been tried.
        """
        from collections import Counter
        position_diversity = {}
        for pos in self.mutable_positions:
            aas_seen = Counter(c["sequence"][pos] for c in population)
            # Entropy-proxy: fewer unique AAs = less explored
            position_diversity[pos] = len(aas_seen)

        # Return positions sorted by least diversity (most underexplored first)
        sorted_positions = sorted(position_diversity.items(), key=lambda x: x[1])
        return [pos for pos, _ in sorted_positions[:top_k]]

    @property
    def size(self):
        return len(self._nodes)


### `Phase 4E · learned_perturbation.py`

In [ ]:
"""
learned_perturbation.py -- Adaptive perturbation operator that improves over time.

Starts with ProteinMPNN's feasibility map as a prior probability distribution.
After each generation, updates per-position, per-residue success rates based
on observed outcomes: did choosing this (position, amino_acid) combination
lead to a candidate that was not dominated by its parent?

Over time the operator learns which substitutions tend to improve the
Pareto front and biases future perturbations toward those moves -- the
perturbations get smarter the longer the run goes.

SCIENTIFIC BASIS:
This is a contextual bandit problem per mutable position, with the
feasibility map as the informed prior and empirical success rates as
the likelihood update. Equivalent to a Bayesian update on a Dirichlet
prior over amino acid choices at each position.
"""
import math, random


class LearnedPerturbation:
    def __init__(self, feasibility_map, mask, mutable_positions,
                 prior_weight=5.0):
        """
        prior_weight: how strongly the feasibility map prior anchors the
        initial distribution. Higher = more rollouts needed to shift away
        from ProteinMPNN's suggestions. 5.0 is a reasonable starting point
        (roughly equivalent to having seen 5 pseudo-observations per residue).
        """
        self.feasibility_map   = feasibility_map
        self.mask              = mask
        self.mutable_positions = mutable_positions
        self.prior_weight      = prior_weight

        # Success/attempt counts per (position, residue): Bayesian update
        # initialized from feasibility map probabilities * prior_weight
        self._successes = {}
        self._attempts  = {}
        for pos in mutable_positions:
            entry = feasibility_map[f"pos_{pos}"]
            for aa, prob in entry["probabilities"].items():
                key = (pos, aa)
                self._successes[key] = prob * prior_weight
                self._attempts[key]  = prior_weight

        self.total_updates = 0

    def _posterior_prob(self, pos, aa):
        """Posterior mean = (successes + prior) / (attempts + prior)."""
        key = (pos, aa)
        s   = self._successes.get(key, 0.0)
        a   = self._attempts.get(key, self.prior_weight)
        return s / a if a > 0 else 0.0

    def _sample_mutation(self, sequence, exclude_positions=None):
        """
        Sample a (position, residue) mutation weighted by posterior probabilities.
        Returns (position, new_residue) or None if no valid mutation found.
        """
        exclude = exclude_positions or set()
        candidates = []
        weights    = []

        for pos in self.mutable_positions:
            if pos in exclude: continue
            entry = self.feasibility_map[f"pos_{pos}"]
            current_aa = sequence[pos]
            for aa in entry["probabilities"]:
                if aa == current_aa: continue
                w = self._posterior_prob(pos, aa)
                if w > 0:
                    candidates.append((pos, aa))
                    weights.append(w)

        if not candidates: return None
        total = sum(weights)
        weights = [w / total for w in weights]
        chosen = random.choices(candidates, weights=weights, k=1)[0]
        return chosen

    def perturb(self, sequence, n_mutations=2):
        """
        Apply n_mutations successive substitutions guided by posterior
        probabilities. Returns mutated sequence.
        """
        result       = list(sequence)
        used_positions = set()

        for _ in range(n_mutations):
            mutation = self._sample_mutation("".join(result), exclude_positions=used_positions)
            if mutation is None: break
            pos, aa = mutation
            result[pos] = aa
            used_positions.add(pos)

        return "".join(result)

    def perturb_toward_unexplored(self, sequence, underexplored_positions, n_mutations=2):
        """
        Biased perturbation: preferentially mutate underexplored positions
        (from FitnessGraph.get_underexplored_positions) to push the search
        into genuinely unvisited regions of sequence space.
        """
        result       = list(sequence)
        used_positions = set()
        n_done       = 0

        # First pass: mutate underexplored positions using posterior
        for pos in underexplored_positions:
            if n_done >= n_mutations: break
            if pos in used_positions: continue
            entry      = self.feasibility_map[f"pos_{pos}"]
            current_aa = result[pos]
            candidates = [(aa, self._posterior_prob(pos, aa))
                          for aa in entry["probabilities"]
                          if aa != current_aa]
            if not candidates: continue
            aas, ws = zip(*candidates)
            total   = sum(ws)
            chosen  = random.choices(aas, weights=[w/total for w in ws], k=1)[0]
            result[pos] = chosen
            used_positions.add(pos)
            n_done += 1

        # Second pass: fill remaining mutations normally
        while n_done < n_mutations:
            mutation = self._sample_mutation("".join(result), exclude_positions=used_positions)
            if mutation is None: break
            pos, aa = mutation
            result[pos] = aa
            used_positions.add(pos)
            n_done += 1

        return "".join(result)

    def update(self, parent_candidate, child_candidate):
        """
        Update success/attempt counts based on observed outcome.
        A mutation is a 'success' if the child is not dominated by the parent
        (i.e., it found a genuinely non-worse solution on at least one axis).
        """

        parent_seq = parent_candidate["sequence"]
        child_seq  = child_candidate["sequence"]

        # Find which positions changed
        changed = [(i, child_seq[i]) for i in self.mutable_positions
                   if parent_seq[i] != child_seq[i]]

        if not changed: return

        # Success: child is not dominated by parent (it's at least as good somewhere)
        success = not dominates(parent_candidate, child_candidate)

        for pos, aa in changed:
            key = (pos, aa)
            self._attempts[key]  = self._attempts.get(key, 0)  + 1.0
            self._successes[key] = self._successes.get(key, 0) + (1.0 if success else 0.0)

        self.total_updates += 1

    def top_learned_mutations(self, n=5):
        """Returns the n highest-posterior (position, residue) pairs -- for audit logging."""
        posteriors = {k: self._posterior_prob(k[0], k[1]) for k in self._successes}
        return sorted(posteriors.items(), key=lambda x: x[1], reverse=True)[:n]


### `Phase 4E · graph_nsga2_engine.py`

In [ ]:
"""
graph_nsga2_engine.py -- Graph-aware NSGA-II with learned perturbation.

Integrates two mechanisms on top of the base NSGA-II:

1. FITNESS LANDSCAPE GRAPH (fitness_graph.py)
   Every evaluated sequence is a node. Edges connect sequences within
   Hamming distance <= 2 at mutable positions. Each generation, after
   survivor selection, the engine checks whether the current population's
   best candidates are local optima in the graph -- i.e., no evaluated
   neighbor dominates them. If so, the search has exhausted the immediate
   neighborhood and needs to jump.

2. LEARNED PERTURBATION (learned_perturbation.py)
   Starts with the feasibility map as a Bayesian prior. After each
   generation, updates per-position success rates based on which
   substitutions produced non-dominated offspring. When a local optimum
   is detected in the graph, fires a targeted perturbation toward
   underexplored graph regions using the learned posterior, instead of
   injecting random sequences (which get outcompeted immediately, as
   we confirmed empirically in run 2 and 3).

DIFFERENCE FROM nsga2_engine.py:
  - Replaces MCTS with LearnedPerturbation as the mutation operator
  - Adds FitnessGraph for topology-aware local minima detection
  - Updates perturbation posterior after every generation (online learning)
  - Perturbations are directed at underexplored graph regions, not random
"""
import os, random, sys


# Re-use NSGA-II primitives from original engine unchanged

# - Hyperparameters -
LOCAL_OPT_THRESHOLD   = 0.5   # fraction of top population that must be local
                               # optima to trigger perturbation (0.5 = majority)
PERTURBATION_SIZE     = 8     # candidates replaced by learned perturbations
PERTURBATION_COOLDOWN = 8     # generations before perturbation can re-fire
N_MUTATIONS_NORMAL    = 2     # mutations per offspring in normal operation
N_MUTATIONS_ESCAPE    = 3     # mutations per offspring during escape event


class GraphNSGA2Engine:
    def __init__(self, feasibility_map, mask, model, baseline_growth,
                 population_size=30, n_generations=50,
                 checkpoint_dir="./phase4_checkpoints/enhanced"):

        self.feasibility_map  = feasibility_map
        self.mask             = mask
        self.model            = model
        self.baseline_growth  = baseline_growth
        self.population_size  = population_size
        self.n_generations    = n_generations
        self.checkpoint_dir   = checkpoint_dir
        self.mutable_positions= get_mutable_positions(mask)

        self.graph      = FitnessGraph(self.mutable_positions, max_edge_distance=2)
        self.perturbation = LearnedPerturbation(feasibility_map, mask,
                                                self.mutable_positions)
        self._fba_cache = {}
        self._cooldown  = 0
        self._escape_count = 0
        os.makedirs(checkpoint_dir, exist_ok=True)

    def _mpb(self, sequence):
        if sequence not in self._fba_cache:
            r = evaluate_candidate(self.model, sequence, self.baseline_growth)
            self._fba_cache[sequence] = r["growth_percent_reduction"]
        return self._fba_cache[sequence]

    def _make_candidate(self, sequence, generation):
        return {
            "sequence":          sequence,
            "structural_score":  compute_structural_score(sequence, self.feasibility_map),
            "mpb_score":         self._mpb(sequence),
            "generation":        generation,
            "rank":              None,
            "crowding_distance": 0.0,
        }

    def _eval_pop(self, sequences, generation):
        return [self._make_candidate(s, generation) for s in sequences]

    def _init_population(self):
        seqs = {WILD_TYPE}
        while len(seqs) < self.population_size:
            seqs.add(sample_sequence(self.feasibility_map, self.mask))
        return list(seqs)

    def _detect_local_optima(self, population):
        """
        Returns True if a majority of the top-ranked population members
        are local optima in the fitness landscape graph.
        """
        # Only check top half (front 0) to avoid triggering on genuinely
        # inferior candidates that are correctly stuck
        top_half = sorted(population, key=lambda c: c.get("rank", 99))
        top_half = top_half[:max(1, len(top_half)//2)]
        local_opt_count = sum(1 for c in top_half if self.graph.is_local_optimum(c))
        return (local_opt_count / len(top_half)) >= LOCAL_OPT_THRESHOLD

    def _fire_escape(self, population, generation):
        """
        Replace PERTURBATION_SIZE worst-ranked candidates with learned
        perturbations directed at underexplored graph regions.
        Unlike random injection, these are posterior-weighted mutations
        at positions the graph identifies as least-explored -- giving
        them a real chance to survive selection pressure.
        """
        underexplored = self.graph.get_underexplored_positions(population, top_k=5)

        escaped_seqs = []
        # Pick diverse parents from the Pareto front to perturb from
        front0 = [c for c in population if c.get("rank", 99) == 0]
        parents = random.choices(front0 if front0 else population,
                                 k=PERTURBATION_SIZE)
        for parent in parents:
            new_seq = self.perturbation.perturb_toward_unexplored(
                parent["sequence"], underexplored, n_mutations=N_MUTATIONS_ESCAPE
            )
            escaped_seqs.append(new_seq)

        escaped = self._eval_pop(escaped_seqs, generation)

        # Replace worst-ranked candidates
        sorted_pop = sorted(population, key=lambda c: (c.get("rank",99),
                                                        -c.get("crowding_distance",0)))
        new_pop = sorted_pop[:-PERTURBATION_SIZE] + escaped
        self._cooldown     = PERTURBATION_COOLDOWN
        self._escape_count += 1
        return new_pop

    def run(self, pareto_archive=None):
        if pareto_archive is None: pareto_archive = ParetoArchive()

        print(f"\n{'='*72}")
        print(f"  Graph-NSGA-II + Learned Perturbation")
        print(f"  pop={self.population_size} | gen={self.n_generations}")
        print(f"  Local optima threshold={LOCAL_OPT_THRESHOLD} | "
              f"Escape size={PERTURBATION_SIZE} | Cooldown={PERTURBATION_COOLDOWN}")
        print(f"{'='*72}")

        population = self._eval_pop(self._init_population(), 0)
        self.graph.add_candidates(population)
        pareto_archive.update(population)
        history = []

        for gen in range(1, self.n_generations + 1):

            # Sort + crowding
            fronts = fast_non_dominated_sort(population)
            for f in fronts: crowding_distance(f, population)

            # Generate offspring via learned perturbation
            offspring_seqs = []
            parent_map     = {}   # child_seq -> parent_candidate for update()
            for _ in range(self.population_size):
                parent  = tournament_select(population)
                n_muts  = N_MUTATIONS_NORMAL
                child_seq = self.perturbation.perturb(parent["sequence"], n_mutations=n_muts)
                offspring_seqs.append(child_seq)
                parent_map[child_seq] = parent

            offspring = self._eval_pop(offspring_seqs, gen)
            self.graph.add_candidates(offspring)

            # Update learned perturbation posterior based on outcomes
            for child in offspring:
                parent = parent_map.get(child["sequence"])
                if parent: self.perturbation.update(parent, child)

            # Survivor selection
            population = select_survivors(population + offspring, self.population_size)
            pareto_archive.update(population)

            # Local optima detection + escape
            escape_tag = ""
            if self._cooldown > 0:
                self._cooldown -= 1
            else:
                fronts2 = fast_non_dominated_sort(population)
                for f in fronts2: crowding_distance(f, population)
                if self._detect_local_optima(population):
                    population = self._fire_escape(population, gen)
                    pareto_archive.update(population)
                    escape_tag = f" [ESCAPE #{self._escape_count} FIRED]"

            best_struct = max(c["structural_score"] for c in population)
            best_mpb    = min(c["mpb_score"]        for c in population)
            top_learned = self.perturbation.top_learned_mutations(n=1)
            top_tag     = f"top_learned=({top_learned[0][0][0]},{top_learned[0][0][1]}:{top_learned[0][1]:.2f})" if top_learned else ""

            history.append({
                "generation":        gen,
                "pareto_front_size": len(pareto_archive.front),
                "cache_size":        len(self._fba_cache),
                "graph_size":        self.graph.size,
                "best_structural":   best_struct,
                "best_mpb":          best_mpb,
                "perturbation_updates": self.perturbation.total_updates,
                "escape_count":      self._escape_count,
            })

            print(f"  gen {gen:03d} | pareto={len(pareto_archive.front):3d} | "
                  f"cache={len(self._fba_cache):4d} | graph={self.graph.size:4d} | "
                  f"struct={best_struct:.3f} | mpb={best_mpb:.4f}% | "
                  f"{top_tag}{escape_tag}")

            if gen % 10 == 0:
                ckpt = os.path.join(self.checkpoint_dir, f"checkpoint_gen_{gen:03d}.json")
                pareto_archive.checkpoint(ckpt, {"generation": gen, "history": history})
                print(f"  [CHECKPOINT] {ckpt}")

        print(f"\n  Done. Pareto={len(pareto_archive.front)} | "
              f"FBA={len(self._fba_cache)} | Graph={self.graph.size} | "
              f"Escapes={self._escape_count} | "
              f"Perturbation updates={self.perturbation.total_updates}")
        return pareto_archive, history


### `Phase 4E · run_phase4_enhanced.py    EXECUTE`

In [ ]:
"""
run_phase4_enhanced.py -- Enhanced Phase 4 orchestrator.

Drops in alongside run_phase4.py. Uses the same Phase 1-3 artifacts
(no re-running anything upstream). Replaces the MCTS + entropy-boost
approach with a graph-structured fitness landscape + learned perturbation.

Run order:
  python run_phase1.py        (already done -- artifacts in ./pipeline_artifacts/)
  python run_phase2.py        (already done)
  python run_phase3.py        (smoke test only -- model loaded fresh here)
  python run_phase4_enhanced.py   <-- this file

Output: ./pipeline_artifacts/phase4_enhanced_pareto_front.json
        (separate from phase4_pareto_front.json so you can compare both)
"""
import sys, os


POPULATION_SIZE = 30
N_GENERATIONS   = 50
FINAL_OUTPUT    = "./pipeline_artifacts/phase4_enhanced_pareto_front.json"


def execute_enhanced_phase_4():
    print("="*72)
    print("  PHASE 4 (ENHANCED): Graph-NSGA-II + Learned Perturbation")
    print("="*72)

    print("\n[1/4] Loading Phase 2 feasibility map and mask...")
    feasibility_map = load_feasibility_map()
    mask            = load_mask()

    print("\n[2/4] Loading iML1515 and computing baseline...")
    model           = load_iml1515()
    baseline_result = evaluate_candidate(model, WILD_TYPE)
    baseline_growth = baseline_result["baseline_growth"]
    wt_mpb          = baseline_result["growth_percent_reduction"]
    print(f"      Baseline growth: {baseline_growth:.4f} 1/h | WT MPB: {wt_mpb:.4f}%")

    print("\n[3/4] Running Graph-NSGA-II + Learned Perturbation...")
    engine = GraphNSGA2Engine(
        feasibility_map = feasibility_map,
        mask            = mask,
        model           = model,
        baseline_growth = baseline_growth,
        population_size = POPULATION_SIZE,
        n_generations   = N_GENERATIONS,
    )
    archive, history = engine.run()

    print("\n[4/4] Saving results...")
    path  = archive.save_final(FINAL_OUTPUT)
    front = archive.front

    print(f"\n{'='*72}")
    print(f"  PHASE 4 ENHANCED COMPLETE")
    print(f"{'='*72}")
    print(f"  Pareto front candidates: {len(front)}")
    print(f"  Wild-type MPB:           {wt_mpb:.4f}%")
    if front:
        best_struct = max(front, key=lambda c: c["structural_score"])
        best_mpb    = min(front, key=lambda c: c["mpb_score"])
        print(f"  Best structural:  {best_struct['structural_score']:.4f} -> {best_struct['sequence']}")
        print(f"  Best MPB:         {best_mpb['mpb_score']:.4f}% -> {best_mpb['sequence']}")
    print(f"  Output: {path}")
    print(f"{'='*72}\n")

    return archive, history



In [ ]:
#  RUN
execute_enhanced_phase_4()

---
## Existence Analysis - Does a Better Solution Provably Exist?
Runs **after** `execute_enhanced_phase_4()` completes.

Four convergent lines of evidence:
1. **Direct check** - does the current Pareto front already beat WT on both axes?
2. **Basin sampling** - Monte Carlo over the ProteinMPNN feasibility prior
3. **IVT / Lipschitz** - intermediate value theorem from evaluated population
4. **GP surrogate** - Gaussian Process P(exists) with 95% credible interval
5. **Biological prior** - homolog position comparison

Requires `scikit-learn` (installed via S1) and `scipy` (optional).
Runtime: ~2 min (GP fit dominates).

In [ ]:
"""
existence_analysis.py
=====================
BioForge - Mathematical Existence Analysis

Answers the question: does a Plectasin variant exist that is BOTH
structurally valid (structural_score > wild-type) AND metabolically
competitive (mpb_score <= wild-type, i.e., no more growth burden)?

Four independent lines of evidence are assembled:

  1. DIRECT PARETO CHECK
     Immediately checks whether any already-evaluated candidate in the
     Pareto front dominates wild-type on both axes. If yes, the question
     is answered empirically - no inference needed.

  2. BASIN SAMPLING (Monte Carlo)
     Draws 10,000 sequences from the ProteinMPNN feasibility-map prior
     and measures what fraction simultaneously score better than WT on
     structure AND are cheaper to produce (via amino acid biosynthetic
     cost as an MPB proxy). Estimates the density of solutions in the
     plausible neighbourhood.

  3. IVT / LIPSCHITZ ARGUMENT
     Uses the Intermediate Value Theorem. If candidate A beats WT on
     structure and candidate B beats WT on MPB, and both fitness
     functions are Lipschitz-continuous (bounded score change per
     mutation), a point along the path A→B must satisfy both. Estimates
     the empirical Lipschitz constant from pairwise score differences in
     the evaluated population.

  4. GAUSSIAN PROCESS SURROGATE
     Fits a GP to the (sequence encoding → score) mapping from all
     available evaluated candidates (aggregated across checkpoints).
     Queries P(struct > WT AND mpb ≤ WT) over the GP posterior via
     Monte Carlo. Gives a probabilistic existence estimate with a 95%
     credible interval.

  5. BIOLOGICAL PRIOR
     Compares optimizer-found mutations against positions known to vary
     in natural Plectasin-family homologs. Independent rediscovery of
     evolutionarily tolerated substitutions is evidence the optimizer is
     finding real biology.

Prerequisites (files that must exist on disk):
  ./pipeline_artifacts/phase2_mask.json
  ./pipeline_artifacts/phase2_feasibility_map.json
  ./pipeline_artifacts/phase4_enhanced_pareto_front.json

Optional (richer GP if present):
  ./phase4_checkpoints/enhanced/checkpoint_gen_*.json

Run order in the notebook:
  - All setup cells (S1-S4)
  - All Phase 1 cells
  - All Phase 2 cells  ← mask + feasibility map written here
  - All Phase 3 cells
  - Phase 4 Enhanced cells + execute_enhanced_phase_4()  ← pareto front written here
  - THIS CELL  (paste below execute_enhanced_phase_4 and call run_existence_analysis())
"""

import os, json, math, random, glob
import numpy as np

# Constants

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX   = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Wild-type chlorotoxin (1CHL chain A, 36 residues)
WILD_TYPE = "GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY"

# WT MPB from the Phase 4 Enhanced run (growth_percent_reduction, %)
# If your run produced a different value, update this.
WT_MPB_PERCENT = 3.5199

# Known positions that vary across natural chlorotoxin-family homologs.
# These are the positions where evolution has already accepted substitutions
HOMOLOG_VARIABLE_POSITIONS = {
    0:  set("ML"),       # M/L - N-terminal variability
    5:  set("FY"),       # F/Y - aromatic, structurally equivalent
    7:  set("TD"),       # T/D - polar substitutions seen
    8:  set("TAS"),      # T/A/S - small polar
    9:  set("DEN"),      # D/E/N - acidic/amide
    12: set("MAL"),      # M/A/L - hydrophobic
    13: set("AR"),       # A/R - flexible position
    24: set("GS"),       # G/S - glycine loop, small residues tolerated
    26: set("KR"),       # K/R - basic, functionally equivalent
    28: set("CY"),       # C/Y - in some homologs C→Y (non-disulfide isoform)
    31: set("PAS"),      # P/A/S - proline bend tolerance
    32: set("QE"),       # Q/E - polar
}

# Amino acid biosynthetic cost in ATP equivalents
# Source: Akashi & Gojobori (2002) PNAS, Table 1
AA_BIOSYNTHETIC_COST = {
    'G': 11.7, 'A': 11.7, 'S': 11.7, 'D': 12.7, 'C': 24.7,
    'T': 18.7, 'P': 20.3, 'E': 15.3, 'Q': 16.2, 'N': 14.7,
    'V': 23.3, 'L': 27.3, 'I': 32.3, 'M': 34.3, 'H': 38.3,
    'F': 52.0, 'Y': 50.0, 'W': 74.3, 'K': 30.3, 'R': 27.3,
}

# Artifact Loading

def _load_json(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[ERROR] {label} not found at: {path}\n"
            f"        Make sure Phase 2 and Phase 4 Enhanced have been run first."
        )
    with open(path) as f:
        return json.load(f)

def load_artifacts():
    """Load mask, feasibility map, Pareto front, and any checkpoint candidates."""
    mask   = _load_json("./pipeline_artifacts/phase2_mask.json",
                        "Phase 2 mask")
    fmap   = _load_json("./pipeline_artifacts/phase2_feasibility_map.json",
                        "Phase 2 feasibility map")
    raw    = _load_json("./pipeline_artifacts/phase4_enhanced_pareto_front.json",
                        "Phase 4 Enhanced Pareto front")

    pareto_front = raw.get("pareto_front", raw if isinstance(raw, list) else [])

    mutable_positions = sorted([
        int(k.split("_")[1])
        for k, v in fmap.items()
        if v.get("status") == "MUTABLE"
    ])

    # Aggregate candidates across all checkpoints for a richer dataset
    all_candidates = list(pareto_front)
    seen_seqs      = {c["sequence"] for c in all_candidates}

    ckpt_dir = "./phase4_checkpoints/enhanced"
    if os.path.isdir(ckpt_dir):
        for ckpt_path in sorted(glob.glob(os.path.join(ckpt_dir, "*.json"))):
            try:
                ckpt = json.loads(open(ckpt_path).read())
                for c in ckpt.get("pareto_front", []):
                    if c.get("sequence") and c["sequence"] not in seen_seqs:
                        all_candidates.append(c)
                        seen_seqs.add(c["sequence"])
            except Exception:
                pass

    print(f"[LOAD] Mutable positions:  {len(mutable_positions)}")
    print(f"[LOAD] Final Pareto front: {len(pareto_front)} candidates")
    print(f"[LOAD] Total unique evaluated (incl. checkpoints): {len(all_candidates)}")

    return mask, fmap, pareto_front, all_candidates, mutable_positions


# Wild-Type Baseline

def compute_wt_scores(fmap, mutable_positions):
    """
    Replicate compute_structural_score(WILD_TYPE, fmap) exactly as the
    engine does - sum of log P(wt_aa | backbone) at mutable positions only.
    """
    wt_struct = 0.0
    for pos in mutable_positions:
        key   = f"pos_{pos}"
        entry = fmap.get(key, {})
        probs = entry.get("probabilities", {})
        aa    = WILD_TYPE[pos] if pos < len(WILD_TYPE) else "G"
        p     = max(probs.get(aa, 1e-10), 1e-10)
        wt_struct += math.log(p)

    wt_mpb = WT_MPB_PERCENT   # from Phase 4 Enhanced run output
    return wt_struct, wt_mpb


# Analysis 0: Direct Pareto Check

def direct_pareto_check(pareto_front, wt_struct, wt_mpb):
    """
    Fastest possible existence proof: does the current Pareto front already
    contain a candidate that beats WT on BOTH axes?
    """
    print("\n" + "=" * 62)
    print("CHECK 0: Direct Pareto Front Inspection")
    print("=" * 62)
    print(f"  Wild-type structural score : {wt_struct:.4f}")
    print(f"  Wild-type MPB              : {wt_mpb:.4f}%")
    print()

    dominates_wt = []
    struct_scores = []
    mpb_scores    = []

    for c in pareto_front:
        s = c.get("structural_score", -999)
        m = c.get("mpb_score", 999)
        struct_scores.append(s)
        mpb_scores.append(m)
        if s > wt_struct and m <= wt_mpb:
            dominates_wt.append(c)

    best_struct = max(struct_scores) if struct_scores else None
    best_mpb    = min(mpb_scores)    if mpb_scores    else None

    print(f"  Best structural score in front : {best_struct:.4f}  "
          f"({'BETTER' if best_struct and best_struct > wt_struct else 'WORSE'} than WT)")
    print(f"  Best MPB in front              : {best_mpb:.4f}%  "
          f"({'BETTER' if best_mpb and best_mpb < wt_mpb else 'WORSE'} than WT)")
    print()

    if dominates_wt:
        print(f"   EMPIRICAL PROOF: {len(dominates_wt)} candidate(s) in the current Pareto")
        print(f"    front already beat wild-type on BOTH objectives.")
        print(f"    The solution exists - it has been found.")
        for c in dominates_wt[:3]:
            print(f"    → seq: {c['sequence']}  "
                  f"struct={c['structural_score']:.3f}  mpb={c['mpb_score']:.4f}%")
    else:
        print(f"   No Pareto candidate dominates WT on both axes yet.")
        print(f"    Struct gap to close:  {wt_struct - (best_struct or wt_struct):.4f} units")
        print(f"    MPB already better:   {wt_mpb - (best_mpb or wt_mpb):.4f}% improvement")
        print(f"    The search is improving on both axes but hasn't converged to")
        print(f"    a single candidate that simultaneously beats WT on both.")

    return {"dominates_wt": dominates_wt, "best_struct": best_struct, "best_mpb": best_mpb}


# Analysis 1: Basin Sampling

def basin_sampling_analysis(fmap, mutable_positions, wt_struct, wt_mpb,
                            n_samples=10_000):
    """
    Monte Carlo over the ProteinMPNN feasibility prior.

    Estimates P(struct > WT  AND  biosynthetic_cost ≤ WT_cost), where cost
    is used as a proxy for MPB because we cannot run FBA on each sample.
    The proxy is conservative - if anything it underestimates overlap,
    since WT's sequence was not selected for E. coli expression efficiency.
    """
    print("\n" + "=" * 62)
    print("ANALYSIS 1: Basin Sampling  (Monte Carlo, n={:,})".format(n_samples))
    print("=" * 62)

    wt_cost = sum(AA_BIOSYNTHETIC_COST.get(aa, 30.0) for aa in WILD_TYPE)

    n_better_struct = 0
    n_cheaper       = 0
    n_both          = 0
    all_struct      = []
    all_cost_ratio  = []

    for _ in range(n_samples):
        seq          = list(WILD_TYPE)
        struct_score = 0.0
        sample_cost  = 0.0

        for pos in mutable_positions:
            entry = fmap.get(f"pos_{pos}", {})
            probs = entry.get("probabilities", {})
            aas   = list(probs.keys())
            ws    = [max(probs[a], 1e-10) for a in aas]
            total = sum(ws)
            ws    = [w / total for w in ws]
            chosen = random.choices(aas, weights=ws, k=1)[0]
            seq[pos] = chosen

        seq_str = "".join(seq)

        # Structural score (same formula as engine)
        for pos in mutable_positions:
            entry = fmap.get(f"pos_{pos}", {})
            probs = entry.get("probabilities", {})
            p     = max(probs.get(seq_str[pos], 1e-10), 1e-10)
            struct_score += math.log(p)

        # Biosynthetic cost (WT-normalised)
        sample_cost = sum(AA_BIOSYNTHETIC_COST.get(aa, 30.0) for aa in seq_str)
        cost_ratio  = sample_cost / wt_cost   # <1.0 means cheaper than WT

        all_struct.append(struct_score)
        all_cost_ratio.append(cost_ratio)

        if struct_score > wt_struct:
            n_better_struct += 1
        if cost_ratio <= 1.0:
            n_cheaper += 1
        if struct_score > wt_struct and cost_ratio <= 1.0:
            n_both += 1

    p_struct = n_better_struct / n_samples
    p_cheap  = n_cheaper       / n_samples
    p_both   = n_both          / n_samples

    mean_struct = float(np.mean(all_struct))
    std_struct  = float(np.std(all_struct))
    mean_ratio  = float(np.mean(all_cost_ratio))

    print(f"  P(structural_score > WT) under feasibility prior : {p_struct:.4f}")
    print(f"  P(biosyn. cost ≤ WT)     under feasibility prior : {p_cheap:.4f}")
    print(f"  P(both simultaneously)                           : {p_both:.4f}  "
          f"({n_both:,} / {n_samples:,} samples)")
    print()
    print(f"  Mean sampled structural score : {mean_struct:.3f}  (σ={std_struct:.3f})")
    print(f"  Mean biosyn. cost ratio vs WT : {mean_ratio:.4f}  "
          f"({'cheaper on average' if mean_ratio < 1 else 'more expensive on average'})")
    print()

    # Independence test: if the two events were uncorrelated,
    # P(both) ≈ P(struct) × P(cheap).  Actual vs expected ratio shows correlation.
    p_if_independent = p_struct * p_cheap
    if p_if_independent > 0:
        lift = p_both / p_if_independent
        print(f"  Correlation lift (actual / if-independent): {lift:.3f}")
        if lift > 1.1:
            print(f"  → The two objectives are POSITIVELY correlated under this prior.")
            print(f"    Sequences that score well structurally also tend to be cheaper.")
        elif lift < 0.9:
            print(f"  → The two objectives are NEGATIVELY correlated (antagonistic).")
        else:
            print(f"  → The two objectives are approximately independent.")

    if n_both > 0:
        print(f"\n   {n_both:,} feasibility-prior samples satisfy both criteria.")
        print(f"    The target region is non-empty - a solution exists in principle.")
    else:
        print(f"\n   No overlap found in {n_samples:,} samples.")
        print(f"    Either objectives are strongly antagonistic, or the proxy is too coarse.")

    return {"p_struct": p_struct, "p_cheap": p_cheap, "p_both": p_both,
            "n_samples": n_samples, "n_both": n_both,
            "mean_struct": mean_struct, "std_struct": std_struct}


# Analysis 2: IVT / Lipschitz Argument

def ivt_lipschitz_argument(all_candidates, wt_struct, wt_mpb):
    """
    Intermediate Value Theorem existence argument.

    If f (structural score) and g (MPB score) are Lipschitz continuous
    in Hamming distance, and there exists:
      - candidate A with f(A) > f(WT)     [better structure]
      - candidate B with g(B) ≤ g(WT)     [better or equal MPB]

    then by IVT there is a point along any Hamming path from A to B where
    both f > f(WT) and g ≤ g(WT) hold simultaneously - provided the score
    functions don't oscillate faster than the Lipschitz bound allows.

    We estimate the empirical Lipschitz constant L from observed pairwise
    score differences / Hamming distances in the evaluated population.
    """
    print("\n" + "=" * 62)
    print("ANALYSIS 2: IVT / Lipschitz Continuity Argument")
    print("=" * 62)

    valid = [c for c in all_candidates
             if c.get("sequence") and len(c["sequence"]) == len(WILD_TYPE)
             and c.get("structural_score") is not None
             and c.get("mpb_score") is not None]

    if not valid:
        print("  [SKIP] No valid candidates available.")
        return {}

    best_struct_cand = max(valid, key=lambda c: c["structural_score"])
    best_mpb_cand    = min(valid, key=lambda c: c["mpb_score"])

    bs = best_struct_cand["structural_score"]
    bm = best_mpb_cand["mpb_score"]

    print(f"  Best structural score : {bs:.4f}  "
          f"(WT: {wt_struct:.4f}  |  diff: {bs - wt_struct:+.4f})")
    print(f"  Best MPB score        : {bm:.4f}%  "
          f"(WT: {wt_mpb:.4f}%  |  diff: {bm - wt_mpb:+.4f}%)")
    print()

    struct_beats_wt = bs > wt_struct
    mpb_beats_wt    = bm < wt_mpb

    # Empirical Lipschitz constant from random pairs
    sample_size = min(300, len(valid))
    sample      = random.sample(valid, sample_size)

    L_struct_obs, L_mpb_obs = [], []
    for i in range(len(sample) - 1):
        a, b = sample[i], sample[i + 1]
        sa, sb = a["sequence"], b["sequence"]
        hamming = sum(x != y for x, y in zip(sa, sb))
        if hamming == 0:
            continue
        L_struct_obs.append(abs(a["structural_score"] - b["structural_score"]) / hamming)
        L_mpb_obs.append(abs(a["mpb_score"] - b["mpb_score"]) / hamming)

    if not L_struct_obs:
        print("  [SKIP] Not enough sequence diversity to estimate Lipschitz constant.")
        return {}

    # Use 90th percentile as a conservative upper bound
    L_struct = float(np.percentile(L_struct_obs, 90))
    L_mpb    = float(np.percentile(L_mpb_obs,    90))

    seq_a = best_struct_cand["sequence"]
    seq_b = best_mpb_cand["sequence"]
    d_AB  = sum(x != y for x, y in zip(seq_a, seq_b))

    # Worst-case score floor along A→B path
    struct_floor = bs  - L_struct * d_AB
    mpb_ceiling  = bm  + L_mpb    * d_AB   # MPB can rise by at most L_mpb * d

    print(f"  Empirical Lipschitz constant (struct) : {L_struct:.4f} score/mutation  [90th pct]")
    print(f"  Empirical Lipschitz constant (MPB)    : {L_mpb:.4f} %/mutation        [90th pct]")
    print(f"  Hamming distance between best-struct and best-MPB : {d_AB}")
    print()
    print(f"  Worst-case structural floor along A→B : {struct_floor:.4f}  (must exceed {wt_struct:.4f})")
    print(f"  Worst-case MPB ceiling  along A→B     : {mpb_ceiling:.4f}%  (must stay ≤ {wt_mpb:.4f}%)")
    print()

    if struct_floor > wt_struct and mpb_ceiling <= wt_mpb:
        verdict = "STRONG"
        msg = ("Both score functions remain above/below WT thresholds along\n"
               "    the entire path. A dual-objective superior sequence provably\n"
               "    exists somewhere between these two candidates.")
    elif struct_beats_wt and mpb_beats_wt:
        verdict = "MODERATE"
        msg = ("Both axes have found candidates beating WT, but the Lipschitz\n"
               "    bound doesn't guarantee scores stay above WT along the full path.\n"
               "    IVT still implies a crossing point; the score may dip briefly\n"
               "    below WT in the interior. A focused local search should find it.")
    elif struct_beats_wt or mpb_beats_wt:
        verdict = "WEAK"
        msg = ("Only one axis currently beats WT. IVT argument is one-sided.\n"
               "    More generations are needed for the other axis to cross WT.")
    else:
        verdict = "NOT MET"
        msg = "Neither axis beats WT yet - IVT conditions not met."

    print(f"  IVT verdict: {verdict}")
    print(f"    {msg}")

    return {
        "struct_beats_wt": struct_beats_wt, "mpb_beats_wt": mpb_beats_wt,
        "L_struct": L_struct, "L_mpb": L_mpb, "d_AB": d_AB,
        "struct_floor": struct_floor, "mpb_ceiling": mpb_ceiling,
        "verdict": verdict,
    }


# Analysis 3: Gaussian Process Surrogate

def gp_surrogate_analysis(all_candidates, mutable_positions, wt_struct, wt_mpb):
    """
    Fits independent GPs to structural_score and mpb_score as functions of
    sequence. Encodes sequences as binary one-hot vectors at mutable positions.

    Queries P(struct > WT AND mpb ≤ WT) across the GP posterior via Monte
    Carlo sampling of the predictive distribution. Returns a point estimate
    and 95% credible interval.
    """
    print("\n" + "=" * 62)
    print("ANALYSIS 3: Gaussian Process Surrogate Model")
    print("=" * 62)

    try:
        from sklearn.gaussian_process import GaussianProcessRegressor
        from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
    except ImportError:
        print("  [SKIP] scikit-learn not installed.  Run:  !pip install scikit-learn")
        return {}

    valid = [c for c in all_candidates
             if c.get("sequence") and len(c["sequence"]) == len(WILD_TYPE)
             and c.get("structural_score") is not None
             and c.get("mpb_score") is not None]

    if len(valid) < 10:
        print(f"  [SKIP] Only {len(valid)} valid candidates - need ≥10 for a meaningful GP.")
        return {}

    print(f"  Training GP on {len(valid)} evaluated sequences...")

    def encode(seq):
        """One-hot encoding at mutable positions → vector of length n_mut × 20."""
        vec = []
        for pos in mutable_positions:
            aa  = seq[pos] if pos < len(seq) else "G"
            vec.extend([1.0 if AMINO_ACIDS[i] == aa else 0.0 for i in range(20)])
        return vec

    X        = np.array([encode(c["sequence"])       for c in valid])
    y_struct = np.array([c["structural_score"]        for c in valid])
    y_mpb    = np.array([c["mpb_score"]               for c in valid])
    X_wt     = np.array([encode(WILD_TYPE)])

    kernel = C(1.0, (1e-3, 1e3)) * Matern(nu=2.5)

    gp_s = GaussianProcessRegressor(kernel=kernel, alpha=0.1,  n_restarts_optimizer=5)
    gp_m = GaussianProcessRegressor(kernel=kernel, alpha=0.01, n_restarts_optimizer=5)

    gp_s.fit(X, y_struct)
    gp_m.fit(X, y_mpb)

    wt_s_pred, wt_s_std = gp_s.predict(X_wt, return_std=True)
    wt_m_pred, wt_m_std = gp_m.predict(X_wt, return_std=True)

    print(f"  GP struct at WT:  predicted {wt_s_pred[0]:.3f} ± {wt_s_std[0]:.3f}  "
          f"(actual: {wt_struct:.3f})")
    print(f"  GP MPB at WT:     predicted {wt_m_pred[0]:.4f} ± {wt_m_std[0]:.4f}%  "
          f"(actual: {wt_mpb:.4f}%)")
    print()

    # Predict across the top-struct candidates (most likely neighbourhood
    # where dual improvement exists)
    top_idx = np.argsort(y_struct)[-min(200, len(valid)):]
    X_test  = X[top_idx]

    mu_s, sig_s = gp_s.predict(X_test, return_std=True)
    mu_m, sig_m = gp_m.predict(X_test, return_std=True)

    # Monte Carlo over GP posterior uncertainty
    n_mc            = 2000
    p_exists_trials = []
    for _ in range(n_mc):
        s_s = mu_s + sig_s * np.random.randn(len(X_test))
        s_m = mu_m + sig_m * np.random.randn(len(X_test))
        p_exists_trials.append(float(np.any((s_s > wt_struct) & (s_m <= wt_mpb))))

    p_exists  = float(np.mean(p_exists_trials))
    ci_lo     = float(np.percentile(p_exists_trials, 2.5))
    ci_hi     = float(np.percentile(p_exists_trials, 97.5))

    print(f"  P(∃ candidate: struct > WT AND mpb ≤ WT) - GP posterior:")
    print(f"    Point estimate : {p_exists:.3f}")
    print(f"    95% CI         : [{ci_lo:.3f},  {ci_hi:.3f}]")
    print()

    if p_exists > 0.7:
        verdict = f" HIGH ({p_exists*100:.0f}%)"
        msg     = "GP strongly supports existence of a dual-superior solution."
    elif p_exists > 0.4:
        verdict = f"~ MODERATE ({p_exists*100:.0f}%)"
        msg     = ("GP uncertainty is high. More evaluations would tighten the CI.\n"
                   "    Existence is plausible but not firmly established yet.")
    else:
        verdict = f" LOW ({p_exists*100:.0f}%)"
        msg     = ("GP doesn't support existence in the tested neighbourhood.\n"
                   "    The dual-objective region may be outside what the current\n"
                   "    population has explored.")

    print(f"  GP verdict: {verdict}")
    print(f"    {msg}")

    return {"p_exists": p_exists, "ci": (ci_lo, ci_hi)}


# Analysis 4: Biological Prior - Homolog Comparison

def biological_prior_analysis(pareto_front, all_candidates, mutable_positions):
    """
    Checks whether the optimizer independently found mutations at positions
    that natural evolution has already accepted in Plectasin-family homologs.

    Independent rediscovery is strong evidence the optimizer is finding
    real structural biology, not just gaming the ProteinMPNN scoring function.

    NOTE: HOMOLOG_VARIABLE_POSITIONS at the top of this file is derived from
    structural alignment literature. Verify entries against UniProt / PDB
    before citing in a paper.
    """
    print("\n" + "=" * 62)
    print("ANALYSIS 4: Biological Prior - Homolog Position Comparison")
    print("=" * 62)
    print("  (Homolog positions are approximate - verify vs UniProt before publishing)")
    print()

    # Gather all mutations found in Pareto front
    opt_mutations = {}   # (pos, aa) → count
    for c in pareto_front:
        seq = c.get("sequence", "")
        if not seq or len(seq) != len(WILD_TYPE):
            continue
        for pos in mutable_positions:
            wt_aa  = WILD_TYPE[pos]
            opt_aa = seq[pos]
            if opt_aa != wt_aa:
                key = (pos, opt_aa)
                opt_mutations[key] = opt_mutations.get(key, 0) + 1

    # Compare against known variable positions
    known_variable = set(HOMOLOG_VARIABLE_POSITIONS.keys()) & set(mutable_positions)
    opt_mut_set    = set(opt_mutations.keys())

    # Rediscovered = optimizer mutated a position that varies in nature
    #              AND chose a residue actually seen in natural homologs
    rediscovered_positions  = set()
    rediscovered_residues   = set()
    novel_positions         = set()

    for (pos, aa), count in opt_mutations.items():
        if pos in HOMOLOG_VARIABLE_POSITIONS:
            rediscovered_positions.add(pos)
            if aa in HOMOLOG_VARIABLE_POSITIONS[pos]:
                rediscovered_residues.add((pos, aa))
        else:
            novel_positions.add(pos)

    print(f"  Unique (position, residue) mutations in Pareto front  : {len(opt_mut_set)}")
    print(f"  Mutable positions overlapping with known variable sites: "
          f"{len(rediscovered_positions)} / {len(known_variable)}")
    print(f"  Optimizer chose a naturally-seen residue at those sites : "
          f"{len(rediscovered_residues)}")
    print(f"  Positions mutated that are novel (not in homolog set)   : "
          f"{len(novel_positions)}")
    print()

    if rediscovered_residues:
        print("  Exact natural-homolog rediscoveries:")
        for pos, aa in sorted(rediscovered_residues):
            count  = opt_mutations.get((pos, aa), 0)
            wt_aa  = WILD_TYPE[pos]
            nat_aa = sorted(HOMOLOG_VARIABLE_POSITIONS[pos] - {wt_aa})
            print(f"    pos {pos:2d}: {wt_aa}→{aa}  "
                  f"(seen in {count}/{len(pareto_front)} Pareto candidates; "
                  f"natural variants at this site: {nat_aa})")
        print()

    # Hypergeometric significance test
    try:
        from scipy.stats import hypergeom
        N = len(mutable_positions) * 19      # total possible (pos,aa) pairs
        K = sum(len(v) - 1 for v in HOMOLOG_VARIABLE_POSITIONS.values()
                if any(p in mutable_positions
                       for p in HOMOLOG_VARIABLE_POSITIONS))  # natural non-WT variants
        n = len(opt_mut_set)                 # optimizer choices
        k = len(rediscovered_residues)       # overlap
        if K > 0 and n > 0:
            p_val = hypergeom.sf(max(k - 1, 0), N, K, n)
            print(f"  Hypergeometric p-value (overlap by chance): {p_val:.4f}")
            if p_val < 0.05:
                print("   Statistically significant overlap (p < 0.05).")
                print("    Optimizer is independently recovering biologically meaningful")
                print("    substitutions, not just fitting the structural scoring function.")
            elif p_val < 0.20:
                print("  ~ Marginal significance (p < 0.20). Suggestive but not conclusive.")
            else:
                print("  ~ Not statistically significant. Overlap may be coincidental,")
                print("    or the homolog set is too small to capture full natural variation.")
    except ImportError:
        print("  [INFO] scipy not available for significance test.  pip install scipy")

    if novel_positions:
        print()
        print(f"  Novel positions (not in homolog variable set): "
              f"{sorted(novel_positions)}")
        print("  These may be genuinely new design space - or artifacts of the")
        print("  ProteinMPNN scoring function. ESMFold validation would clarify.")

    return {
        "opt_mutations":          opt_mutations,
        "rediscovered_positions": rediscovered_positions,
        "rediscovered_residues":  rediscovered_residues,
        "novel_positions":        novel_positions,
    }


# Summary Report

def print_summary(direct, basin, ivt, gp, bio):
    print()
    print("=" * 62)
    print("  EXISTENCE ANALYSIS - SUMMARY REPORT")
    print("=" * 62)
    print()
    print("  QUESTION: Does a Plectasin variant exist that simultaneously")
    print("  beats wild-type on structural score AND metabolic burden?")
    print()

    evidence = []

    # Direct check
    if direct.get("dominates_wt"):
        n = len(direct["dominates_wt"])
        evidence.append(("", f"DIRECT:    {n} Pareto candidate(s) already beat WT on both axes."))
    else:
        bs = direct.get("best_struct")
        bm = direct.get("best_mpb")
        if bs is not None and bm is not None:
            evidence.append(("~", f"DIRECT:    No single candidate beats WT on both yet "
                                   f"(struct gap: {abs(bs):.1f} vs WT {abs(direct.get('wt_struct', bs)):.1f}; "
                                   f"MPB already {bm:.2f}% vs WT)."))
        else:
            evidence.append(("", "DIRECT:    Could not evaluate (missing score data)."))

    # Basin sampling
    p_both = basin.get("p_both", 0)
    n_both = basin.get("n_both", 0)
    ns     = basin.get("n_samples", 0)
    if p_both > 0:
        evidence.append(("", f"SAMPLING:  {n_both:,}/{ns:,} feasibility-prior samples "
                               f"satisfy both criteria (p={p_both:.4f})."))
    else:
        evidence.append(("", f"SAMPLING:  No overlap in {ns:,} Monte Carlo samples."))

    # IVT
    v = ivt.get("verdict", "NOT MET")
    if v == "STRONG":
        evidence.append(("", f"IVT:       Strong - scores stay above/below WT along entire A→B path."))
    elif v == "MODERATE":
        evidence.append(("~", f"IVT:       Moderate - both axes beat WT individually; "
                               "crossing point implied."))
    elif v == "WEAK":
        evidence.append(("~", f"IVT:       Weak - only one axis beats WT so far."))
    else:
        evidence.append(("", f"IVT:       Conditions not met."))

    # GP
    p_ex = gp.get("p_exists")
    if p_ex is not None:
        ci = gp.get("ci", (0, 1))
        if p_ex > 0.7:
            evidence.append(("", f"GP:        P(exists) = {p_ex:.2f}  "
                                   f"[95% CI {ci[0]:.2f}-{ci[1]:.2f}]  - high confidence."))
        elif p_ex > 0.4:
            evidence.append(("~", f"GP:        P(exists) = {p_ex:.2f}  "
                                   f"[95% CI {ci[0]:.2f}-{ci[1]:.2f}]  - moderate confidence."))
        else:
            evidence.append(("", f"GP:        P(exists) = {p_ex:.2f}  - low confidence."))
    else:
        evidence.append(("~", "GP:        Skipped (scikit-learn not available or insufficient data)."))

    # Biological
    nr = len(bio.get("rediscovered_residues", set()))
    if nr > 0:
        evidence.append(("", f"BIOLOGY:   Optimizer independently rediscovered {nr} naturally-"
                               "occurring substitution(s) from homolog alignment."))
    else:
        evidence.append(("~", "BIOLOGY:   No exact natural-homolog rediscoveries detected."))

    n_positive = sum(1 for mark, _ in evidence if mark == "")

    for mark, text in evidence:
        print(f"  {mark}  {text}")

    print()
    print(f"  Lines of evidence: {n_positive}/{len(evidence)} positive")
    print()

    if n_positive >= 4:
        print("  -")
        print("  CONCLUSION: Strong convergent evidence.")
        print("  A structurally valid, metabolically competitive variant")
        print("  of Plectasin almost certainly exists. The optimizer")
        print("  may have already found one - inspect the Pareto front")
        print("  direct check above. If not yet, it is very close.")
        print("  Recommended: run ESMFold on the top 5 Pareto candidates")
        print("  to confirm the structural score proxy reflects real fold.")
        print("  -")
    elif n_positive >= 2:
        print("  -")
        print("  CONCLUSION: Moderate evidence.")
        print("  The solution likely exists but hasn't been pinpointed.")
        print("  Consider: larger population (pop=60+), more generations,")
        print("  or adding MPB >= WT as a hard constraint rather than a")
        print("  free objective to force the optimizer into that region.")
        print("  -")
    else:
        print("  -")
        print("  CONCLUSION: Weak evidence. The objectives may be more")
        print("  antagonistic than expected, or the structural score")
        print("  proxy is drifting from actual fold quality. ESMFold")
        print("  validation and objective rebalancing are recommended.")
        print("  -")
    print()


# Entry Point

def run_existence_analysis(seed=42):
    print("=" * 62)
    print("  BioForge - Existence Analysis")
    print("  Plectasin Structural + Metabolic Optimization")
    print("=" * 62)

    random.seed(seed)
    np.random.seed(seed)

    mask, fmap, pareto_front, all_candidates, mutable_positions = load_artifacts()
    wt_struct, wt_mpb = compute_wt_scores(fmap, mutable_positions)

    print(f"\n  Wild-type structural score : {wt_struct:.4f}")
    print(f"  Wild-type MPB              : {wt_mpb:.4f}%")

    direct = direct_pareto_check(pareto_front, wt_struct, wt_mpb)
    # Patch wt_struct into direct dict for summary
    direct["wt_struct"] = wt_struct

    basin  = basin_sampling_analysis(fmap, mutable_positions, wt_struct, wt_mpb)
    ivt    = ivt_lipschitz_argument(all_candidates, wt_struct, wt_mpb)
    gp     = gp_surrogate_analysis(all_candidates, mutable_positions, wt_struct, wt_mpb)
    bio    = biological_prior_analysis(pareto_front, all_candidates, mutable_positions)

    print_summary(direct, basin, ivt, gp, bio)


run_existence_analysis()


---
## Benchmark - Random Mutation Baseline
Run AFTER both Phase 4 runs. Produces the control result for comparison.

In [ ]:
"""
benchmark_random_baseline.py -- Identical NSGA-II run with random mutation.

Produces phase4_pareto_front_RANDOM.json alongside the MCTS run's output.
Run AFTER run_phase4.py so the FBA cache from the MCTS run warms up this
one too (they share the same underlying model and sequence space, so many
sequences will be re-encountered and returned from cache instantly).

Compare the two Pareto fronts using validators/validate_pareto_visualizer.py
to demonstrate whether MCTS-guided mutation improved Pareto front quality.
"""
import sys, os


POPULATION_SIZE = 30
N_GENERATIONS   = 50
OUTPUT_PATH     = "./pipeline_artifacts/phase4_pareto_front_RANDOM.json"


def run_random_baseline():
    print("="*72)
    print("  BENCHMARK: NSGA-II with RANDOM mutation (no MCTS)")
    print("="*72)

    feasibility_map = load_feasibility_map()
    mask            = load_mask()
    model           = load_iml1515()
    baseline_result = evaluate_candidate(model, WILD_TYPE)
    baseline_growth = baseline_result["baseline_growth"]

    engine = NSGA2Engine(
        feasibility_map = feasibility_map,
        mask            = mask,
        model           = model,
        baseline_growth = baseline_growth,
        population_size = POPULATION_SIZE,
        n_generations   = N_GENERATIONS,
        checkpoint_dir  = "./phase4_checkpoints/random_baseline",
        use_mcts        = False,   # <-- the only difference from run_phase4.py
    )
    archive, history = engine.run()
    archive.save_final(OUTPUT_PATH)

    print(f"\n  Random baseline Pareto front: {len(archive.front)} candidates")
    print(f"  Saved to: {OUTPUT_PATH}")
    return archive, history



In [ ]:
#  RUN
run_random_baseline()

---
## Validators - Analysis & Visualization
Run after all Phase 4 runs are complete.

### `validate_pareto_visualizer.py`

In [ ]:
"""
validate_pareto_visualizer.py -- Pareto frontier plots and hypervolume tracking.

Produces:
  1. 2D scatter: structural_score vs mpb_score for MCTS vs random Pareto fronts
  2. Per-generation hypervolume improvement chart (if history JSON provided)
  3. Wild-type 1ZFU (Plectasin) marked as reference point on both plots

Requires: pip install matplotlib pymoo
"""
import sys, os, json, math

import matplotlib
matplotlib.use("Agg")   # headless-safe; change to "TkAgg" or remove for interactive
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


MCTS_FRONT_PATH   = "./pipeline_artifacts/phase4_pareto_front.json"
RANDOM_FRONT_PATH = "./pipeline_artifacts/phase4_pareto_front_RANDOM.json"
OUTPUT_DIR        = "./pipeline_artifacts/figures"


def load_front(path):
    with open(path) as f: return json.load(f)["pareto_front"]


def compute_hypervolume_2d(front, ref_point):
    """
    2D hypervolume indicator. ref_point = (worst_structural, worst_mpb).
    Structural: maximize, so flip sign for standard min-based computation.
    """
    # Convert to minimization: negate structural_score
    points = sorted([(-c["structural_score"], c["mpb_score"]) for c in front],
                    key=lambda p: p[0])
    hv, prev_x = 0.0, ref_point[0]
    for x, y in points:
        if y < ref_point[1]:
            hv += (ref_point[0] - x) * (ref_point[1] - y)
    return hv


def plot_pareto_comparison(mcts_front, random_front, wt_structural, wt_mpb, out_dir):
    fig, ax = plt.subplots(figsize=(9, 6))

    def _plot_front(front, color, label, marker):
        xs = [c["structural_score"] for c in front]
        ys = [c["mpb_score"]        for c in front]
        ax.scatter(xs, ys, c=color, label=label, marker=marker, s=60, alpha=0.85, zorder=3)

    if mcts_front:   _plot_front(mcts_front,   "#2196F3", "MCTS-guided NSGA-II", "o")
    if random_front: _plot_front(random_front, "#FF5722", "Random mutation NSGA-II", "^")

    # Wild-type reference
    ax.scatter([wt_structural], [wt_mpb], c="#4CAF50", marker="*", s=250,
               label=f"Wild-type 1ZFU (Plectasin)", zorder=5)
    ax.annotate("WT", (wt_structural, wt_mpb), textcoords="offset points",
                xytext=(6, 4), fontsize=8, color="#4CAF50")

    ax.set_xlabel("Structural Viability Score (log P, ↑ higher = better)", fontsize=11)
    ax.set_ylabel("Metabolic Production Burden (% growth reduction, ↓ lower = better)", fontsize=11)
    ax.set_title("Pareto Frontier: Structure vs. Manufacturability\n(Plectasin (1ZFU) Peptide Optimization)", fontsize=12)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "pareto_comparison.png")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    print(f"[SAVED] {path}")
    return path


def plot_hypervolume_history(history_path, out_dir):
    """Plots per-generation hypervolume from a checkpoint metadata JSON."""
    if not os.path.exists(history_path):
        print(f"[SKIP] No history file at {history_path}")
        return

    with open(history_path) as f: data = json.load(f)
    history = data.get("metadata", {}).get("history", [])
    if not history: print("[SKIP] No history data in checkpoint."); return

    gens  = [h["generation"]       for h in history]
    sizes = [h["pareto_front_size"] for h in history]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gens, sizes, color="#2196F3", linewidth=2)
    ax.fill_between(gens, sizes, alpha=0.15, color="#2196F3")
    ax.set_xlabel("Generation"); ax.set_ylabel("Pareto Front Size")
    ax.set_title("Pareto Front Growth Over Generations")
    ax.grid(True, alpha=0.3)

    path = os.path.join(out_dir, "pareto_front_growth.png")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    print(f"[SAVED] {path}")
    return path


def run_visualization():
    print("="*60)
    print("  VALIDATOR: Pareto Frontier Visualization")
    print("="*60)

    fmap         = load_feasibility_map()
    wt_structural = compute_structural_score(WILD_TYPE, fmap)

    mcts_front   = load_front(MCTS_FRONT_PATH)   if os.path.exists(MCTS_FRONT_PATH)   else []
    random_front = load_front(RANDOM_FRONT_PATH) if os.path.exists(RANDOM_FRONT_PATH) else []

    # Wild-type MPB from the saved front (it's always included in initial pop)
    wt_entries = [c for c in mcts_front if c["sequence"] == WILD_TYPE]
    wt_mpb = wt_entries[0]["mpb_score"] if wt_entries else 3.52   # Phase 3 confirmed value

    plot_pareto_comparison(mcts_front, random_front, wt_structural, wt_mpb, OUTPUT_DIR)

    # Latest checkpoint for history
    ckpt_dir = "./phase4_checkpoints"
    if os.path.isdir(ckpt_dir):
        ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith(".json")])
        if ckpts:
            plot_hypervolume_history(os.path.join(ckpt_dir, ckpts[-1]), OUTPUT_DIR)

    print(f"\n  MCTS front:   {len(mcts_front)} candidates")
    print(f"  Random front: {len(random_front)} candidates")
    print(f"  Figures saved to: {OUTPUT_DIR}")



In [ ]:
#  RUN
run_visualization()

### `validate_sequence_analyzer.py`

In [ ]:
"""
validate_sequence_analyzer.py -- Analyzes final Pareto-front candidates.

Produces per-candidate breakdown:
  - Which positions mutated vs wild-type
  - Amino acid composition shift
  - Basic physicochemical properties (charge, hydrophobicity, MW)
  - Mask compliance check (no frozen positions mutated)
  - Summary CSV for paper tables

Requires: pip install pandas
"""
import sys, os, json

import pandas as pd

MCTS_FRONT_PATH = "./pipeline_artifacts/phase4_pareto_front.json"
OUTPUT_DIR      = "./pipeline_artifacts"

# Kyte-Doolittle hydrophobicity scale
HYDROPHOBICITY = {
    'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
    'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
    'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2
}
# Charge at pH 7.4 (simplified: R,K=+1; D,E=-1; H=+0.1)
CHARGE = {'R':1.0,'K':1.0,'H':0.1,'D':-1.0,'E':-1.0}

# Approximate residue molecular weights (Da)
MW = {
    'A':89,'R':174,'N':132,'D':133,'C':121,'Q':146,'E':147,'G':75,
    'H':155,'I':131,'L':131,'K':146,'M':149,'F':165,'P':115,'S':105,
    'T':119,'W':204,'Y':181,'V':117
}

def sequence_properties(seq):
    hydro  = sum(HYDROPHOBICITY.get(aa,0) for aa in seq) / len(seq)
    charge = sum(CHARGE.get(aa,0) for aa in seq)
    mw     = sum(MW.get(aa,0) for aa in seq)
    return {"hydrophobicity": round(hydro,3),
            "net_charge":     round(charge,2),
            "molecular_weight_Da": mw}

def find_mutations(sequence, wild_type=WILD_TYPE):
    return [f"{wt}{i+1}{aa}"
            for i,(aa,wt) in enumerate(zip(sequence, wild_type))
            if aa != wt]

def run_sequence_analysis():
    print("="*60)
    print("  VALIDATOR: Sequence Analysis")
    print("="*60)

    with open(MCTS_FRONT_PATH) as f:
        data = json.load(f)
    front = data["pareto_front"]
    mask  = load_mask()

    rows = []
    violations_found = False

    for cand in front:
        seq   = cand["sequence"]
        valid, violations = validate_sequence(seq, mask)
        if not valid:
            print(f"[WARNING] Mask violation in candidate: {violations}")
            violations_found = True

        mutations = find_mutations(seq)
        props     = sequence_properties(seq)
        rows.append({
            "sequence":         seq,
            "structural_score": round(cand["structural_score"], 4),
            "mpb_score":        round(cand["mpb_score"], 4),
            "generation":       cand.get("generation","?"),
            "n_mutations":      len(mutations),
            "mutations":        ", ".join(mutations) if mutations else "wild-type",
            "mask_valid":       valid,
            **props,
        })

    df = pd.DataFrame(rows).sort_values("mpb_score")

    csv_path = os.path.join(OUTPUT_DIR, "phase4_pareto_analysis.csv")
    df.to_csv(csv_path, index=False)

    print(f"\n  Pareto front candidates: {len(df)}")
    print(f"  Mask violations: {'NONE' if not violations_found else 'SEE ABOVE'}")
    print(f"\n  Top 5 by lowest MPB (cheapest to manufacture):")
    print(df[["mutations","structural_score","mpb_score","net_charge","hydrophobicity"]].head().to_string(index=False))
    print(f"\n  Top 5 by highest structural score:")
    print(df.sort_values("structural_score",ascending=False)[
        ["mutations","structural_score","mpb_score","net_charge","hydrophobicity"]
    ].head().to_string(index=False))
    print(f"\n  Full table saved to: {csv_path}")

    return df



In [ ]:
#  RUN
run_sequence_analysis()

### `validate_esm_fold.py`
  GPU required. Enable T4 GPU in Runtime → Change runtime type before running.

In [ ]:
"""
validate_esm_fold.py -- ESMFold structural validation for Pareto-front candidates.

Runs only on elite candidates (top N by combined rank from the Pareto front)
to confirm that optimized mutations actually preserve the Plectasin fold.

RESOURCE WARNING: ESMFold requires ~16GB GPU VRAM for the full model.
On Colab free tier (T4, ~16GB VRAM), this is borderline. If it OOMs:
  1. Reduce MAX_CANDIDATES to 3.
  2. Use the ESMFold API instead: https://esmatlas.com/resources?action=fold
  3. Use a lighter folding model (OmegaFold or HelixFold-Single via API).

Requires: pip install transformers torch
"""
import sys, os, json

import torch

MCTS_FRONT_PATH = "./pipeline_artifacts/phase4_pareto_front.json"
OUTPUT_PATH     = "./pipeline_artifacts/phase4_esm_validation.json"
MAX_CANDIDATES  = 5    # lower if OOM; these are expensive


def load_esm_fold():
    """Loads ESMFold via HuggingFace transformers. Downloads ~2.7GB on first run."""
    from transformers import EsmForProteinFolding, EsmTokenizer
    print("[INFO] Loading ESMFold (first run downloads ~2.7GB)...")
    tokenizer = EsmTokenizer.from_pretrained("facebook/esmfold_v1")
    model     = EsmForProteinFolding.from_pretrained(
        "facebook/esmfold_v1", low_cpu_mem_usage=True
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("[WARNING] No GPU found. ESMFold on CPU will be very slow (~10 min/sequence).")
    model = model.to(device)
    model.eval()
    print(f"[INFO] ESMFold loaded on {device}.")
    return model, tokenizer, device


def fold_sequence(sequence, model, tokenizer, device):
    """Returns pLDDT (per-residue confidence) and mean pLDDT for one sequence."""
    inputs  = tokenizer([sequence], return_tensors="pt", add_special_tokens=False)
    inputs  = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    plddt = outputs.plddt.squeeze().cpu().numpy()   # shape [L]
    return {
        "mean_plddt": float(plddt.mean()),
        "min_plddt":  float(plddt.min()),
        "plddt_per_residue": plddt.tolist(),
    }


def select_elite_candidates(front, n=MAX_CANDIDATES):
    """
    Selects top N candidates covering the Pareto front:
    best structural, best MPB, and spread of trade-offs between them.
    """
    if len(front) <= n: return front
    srt_struct = sorted(front, key=lambda c: c["structural_score"], reverse=True)
    srt_mpb    = sorted(front, key=lambda c: c["mpb_score"])
    elite = [srt_struct[0], srt_mpb[0]]
    # Fill remaining slots with evenly-spaced candidates by MPB
    step = max(1, len(front) // (n - 2))
    for i in range(step, len(front), step):
        if len(elite) >= n: break
        if front[i] not in elite: elite.append(front[i])
    return elite[:n]


def run_esm_validation():
    print("="*60)
    print("  VALIDATOR: ESMFold Structural Validation")
    print("="*60)

    with open(MCTS_FRONT_PATH) as f:
        data = json.load(f)
    front = data["pareto_front"]

    elite     = select_elite_candidates(front, MAX_CANDIDATES)
    model, tokenizer, device = load_esm_fold()

    results = []

    # Always validate wild-type first as structural reference baseline
    all_seqs = [{"sequence": WILD_TYPE, "is_wild_type": True,
                 "structural_score": None, "mpb_score": None}]
    for c in elite:
        all_seqs.append({"sequence": c["sequence"], "is_wild_type": False,
                          "structural_score": c["structural_score"],
                          "mpb_score": c["mpb_score"]})

    for entry in all_seqs:
        seq  = entry["sequence"]
        tag  = "WILD-TYPE" if entry["is_wild_type"] else "CANDIDATE"
        print(f"\n[FOLDING] {tag}: {seq}")
        fold = fold_sequence(seq, model, tokenizer, device)
        result = {**entry, **fold}
        results.append(result)
        print(f"  mean pLDDT: {fold['mean_plddt']:.2f} | min pLDDT: {fold['min_plddt']:.2f}")

    # Save
    with open(OUTPUT_PATH, "w") as f:
        json.dump({"esm_validations": results}, f, indent=2)

    # Summary
    wt_plddt = next(r["mean_plddt"] for r in results if r["is_wild_type"])
    print(f"\n{'='*60}")
    print(f"  Wild-type mean pLDDT: {wt_plddt:.2f}")
    print(f"  (pLDDT >= 70 = confident; >= 90 = very high confidence)")
    for r in results:
        if not r["is_wild_type"]:
            delta = r["mean_plddt"] - wt_plddt
            flag  = "" if r["mean_plddt"] >= 70 else ""
            print(f"  {flag} candidate pLDDT={r['mean_plddt']:.2f} (Δ{delta:+.2f}) | "
                  f"mpb={r['mpb_score']:.4f}%")
    print(f"\n  Results saved to: {OUTPUT_PATH}")
    print(f"{'='*60}")

    return results



In [ ]:
#  RUN (GPU required - uncomment pip install if needed)
# !pip install transformers
run_esm_validation()

---
## Phase 5 - Divergence Analysis: FBA vs Physicochemical Proxy
**The make-or-break experiment.**

Runs NSGA-II with the same structural objective but replaces the FBA metabolic cost
objective with a physicochemical proxy (instability index + GRAVY score) - the
approach used by Yang et al. 2024 and the broader AMP design literature.

If FBA and the proxy select substantially different Pareto fronts - especially if
FBA-selected sequences are systematically lower in expensive amino acids (W, Y, F, M, H)
while proxy-selected sequences are not - that divergence is the central empirical
result justifying the framework as a novel contribution.

Run all 3 cells below **after** Phase 4 has completed.

In [ ]:
"""
proxy_evaluator.py
------------------
Phase 5.1: Physicochemical Proxy Cost Evaluator.

Drop-in replacement for fba_evaluator.py that computes manufacturability
via physicochemical proxies only - no FBA, no metabolic model.

Proxy cost = composite of:
  - Instability Index  (Guruprasad et al. 1990; lower = more stable)
  - GRAVY score        (Kyte-Doolittle hydropathicity; higher = more hydrophobic)

This mirrors the objective function used by Yang et al. (2024)
in the closest related published work.
"""

from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np

# E. coli biosynthetic cost per amino acid (ATP equivalents, Akashi & Gojobori 2002)
# Used for post-hoc annotation only - NOT used as an objective in proxy run.
AA_BIOSYNTHETIC_COST = {
    'A':  11.7, 'R':  27.3, 'N':  14.7, 'D':  12.8, 'C':  24.7,
    'Q':  16.3, 'E':  15.3, 'G':  11.7, 'H':  38.8, 'I':  32.7,
    'L':  27.3, 'K':  30.3, 'M':  34.3, 'F':  52.0, 'P':  20.3,
    'S':  18.7, 'T':  18.7, 'W':  74.3, 'Y':  50.0, 'V':  23.3,
}
EXPENSIVE_AA = {'W', 'Y', 'F', 'M', 'H', 'C', 'R'}  # top 7 by cost


class ProxyEvaluator:
    """
    Computes proxy manufacturability score from physicochemical properties.
    Returns a cost in [0, 100] - lower is better (cheaper to manufacture
    in proxy terms). Designed to be interface-compatible with FBAEvaluator.
    """

    def __init__(self):
        self._cache = {}
        self._calls = 0

    def compute_proxy_cost(self, sequence: str) -> dict:
        """Compute proxy cost and component scores for a sequence."""
        if sequence in self._cache:
            return self._cache[sequence]

        self._calls += 1
        analysis = ProteinAnalysis(sequence)

        instability = analysis.instability_index()   # 0-100+, lower = more stable
        gravy       = analysis.gravy()                # negative = hydrophilic
        aliphatic   = _aliphatic_index(sequence)      # higher = more aliphatic

        # Proxy cost: penalise instability and hydrophobicity.
        # Normalised so typical values land in [0, 100].
        proxy_cost = (
            0.5 * np.clip(instability, 0, 100)
            + 0.5 * np.clip((gravy + 2.0) * 25, 0, 100)  # shift GRAVY ~[-2,2] → [0,100]
        )

        # True biosynthetic cost for annotation (not used as objective)
        true_cost = sum(AA_BIOSYNTHETIC_COST.get(aa, 20.0) for aa in sequence)
        expensive_count = sum(1 for aa in sequence if aa in EXPENSIVE_AA)

        result = {
            'proxy_cost':        float(proxy_cost),
            'instability_index': float(instability),
            'gravy':             float(gravy),
            'aliphatic_index':   float(aliphatic),
            'true_biosyn_cost':  float(true_cost),
            'expensive_aa_count': expensive_count,
        }
        self._cache[sequence] = result
        return result

    def score(self, sequence: str) -> float:
        """Return proxy cost scalar (interface-compatible with FBAEvaluator)."""
        return self.compute_proxy_cost(sequence)['proxy_cost']


def _aliphatic_index(sequence: str) -> float:
    """Ikai 1980 aliphatic index."""
    n  = len(sequence)
    xa = sequence.count('A') / n
    xv = sequence.count('V') / n
    xi = sequence.count('I') / n
    xl = sequence.count('L') / n
    return 100 * (xa + 2.9 * xv + 3.9 * (xi + xl))


print(' ProxyEvaluator defined.')
print(f'   Expensive AAs tracked: {", ".join(sorted(EXPENSIVE_AA))}')
print(f'   Most expensive: W ({AA_BIOSYNTHETIC_COST["W"]:.1f} ATP eq)')
print(f'   Cheapest: G/A ({AA_BIOSYNTHETIC_COST["G"]:.1f} ATP eq)')


In [ ]:
"""
run_phase5_proxy.py
-------------------
Phase 5.2: Proxy-objective NSGA-II run.

Identical to Phase 4 (NSGA-II + MCTS) except the FBA metabolic cost
objective is replaced by the physicochemical proxy cost from
ProxyEvaluator. All other parameters (population, generations, mask,
feasibility map, MCTS operator) are held constant so that any
difference in the resulting Pareto front is attributable solely to
the choice of cost objective.
"""

import json, os, copy, time
import numpy as np

PROXY_OUTPUT_PATH = './pipeline_artifacts/phase5_proxy_pareto_front.json'
PROXY_CHECKPOINT_DIR = './phase5_checkpoints/proxy'
os.makedirs(PROXY_CHECKPOINT_DIR, exist_ok=True)


def execute_pipeline_phase_5_proxy():
    print('=' * 72)
    print('  PHASE 5: PROXY NSGA-II RUN (Physicochemical Cost Objective)')
    print('=' * 72)
    print('  Objective 1: Structural score  (ProteinMPNN - same as Phase 4)')
    print('  Objective 2: Proxy cost        (instability index + GRAVY - NO FBA)')
    print('  All other params identical to Phase 4.')
    print()

    # [1/4] Load feasibility map and mask (same as Phase 4)
    print('[1/4] Loading Phase 2 feasibility map and mask...')
    with open('./pipeline_artifacts/phase2_feasibility_map.json') as f:
        feasibility_map = json.load(f)
    with open('./pipeline_artifacts/phase2_mask.json') as f:
        mask = json.load(f)

    WILD_TYPE = 'GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY'
    mutable_positions = sorted(
        int(k.split('_')[1]) for k, v in mask.items() if v['status'] == 'MUTABLE'
    )
    print(f'      Mutable positions: {len(mutable_positions)}')

    # [2/4] Initialise proxy evaluator (no model loading needed)
    print('[2/4] Initialising proxy evaluator (no metabolic model)...')
    proxy_eval = ProxyEvaluator()
    wt_proxy = proxy_eval.score(WILD_TYPE)
    print(f'      Wild-type proxy cost: {wt_proxy:.4f}')

    # [3/4] Run NSGA-II with proxy objective
    print('[3/4] Running proxy NSGA-II...')
    print()
    print('=' * 72)
    print('  Proxy-NSGA-II [MCTS] | pop=30 | gen=50')
    print('=' * 72)

    mcts_op   = MCTSOperator(feasibility_map, mask)
    archive   = ParetoArchive()
    history   = []
    cache     = {}

    def evaluate_proxy(seq):
        if seq in cache:
            return cache[seq]
        struct = _compute_structural_score(seq, feasibility_map, mutable_positions, WILD_TYPE)
        cost   = proxy_eval.score(seq)
        result = {'sequence': seq, 'structural_score': struct, 'mpb_score': cost,
                  'proxy_cost': cost, 'is_proxy': True}
        cache[seq] = result
        return result

    # Initialise population
    population = []
    seen = set()
    while len(population) < 30:
        seq = _random_feasible(WILD_TYPE, mutable_positions, feasibility_map)
        if seq not in seen:
            seen.add(seq)
            population.append(evaluate_proxy(seq))

    stagnation_counter = 0
    best_struct_prev   = max(c['structural_score'] for c in population)
    boost_gens_left    = 0
    total_boosts       = 0

    for gen in range(1, 51):
        # MCTS offspring
        rollouts = 150 if boost_gens_left > 0 else 30
        offspring = []
        mcts_op.n_rollouts = rollouts
        for parent in population:
            seq = mcts_op.mutate(parent['sequence'])
            if seq not in seen:
                seen.add(seq)
                offspring.append(evaluate_proxy(seq))

        combined  = population + offspring
        fronts    = fast_non_dominated_sort(combined)
        next_pop  = []
        for front in fronts:
            if len(next_pop) + len(front) <= 30:
                next_pop.extend(front)
            else:
                needed = 30 - len(next_pop)
                ranked = crowding_distance_sort(front)
                next_pop.extend(ranked[:needed])
                break
        population = next_pop

        for c in population:
            archive.update(c)

        best_struct = max(c['structural_score'] for c in population)
        best_cost   = min(c['mpb_score'] for c in population)
        entropy     = _sequence_entropy(population, mutable_positions)

        if abs(best_struct - best_struct_prev) < 0.01:
            stagnation_counter += 1
        else:
            stagnation_counter = 0
        best_struct_prev = best_struct

        boost_fired = False
        if stagnation_counter >= 5 and entropy < 1.5 and boost_gens_left == 0:
            boost_gens_left = 5
            total_boosts   += 1
            stagnation_counter = 0
            boost_fired = True
        if boost_gens_left > 0:
            boost_gens_left -= 1

        boost_tag = f' [BOOST #{total_boosts} FIRED]' if boost_fired else \
                    (f' [boost {boost_gens_left}gen left]' if boost_gens_left > 0 else '')
        print(f'  gen {gen:03d} | pareto={len(archive.front):3d} | cache={len(cache):4d} | '
              f'struct={best_struct:.3f} | proxy={best_cost:.4f} | H={entropy:.2f}b{boost_tag}')

        history.append({'generation': gen, 'pareto_front_size': len(archive.front),
                        'cache_size': len(cache), 'best_structural': best_struct,
                        'best_proxy_cost': best_cost, 'entropy_bits': entropy})

        if gen % 10 == 0:
            ckpt = {'generation': gen, 'pareto_front': archive.front, 'history': history}
            with open(f'{PROXY_CHECKPOINT_DIR}/checkpoint_gen_{gen:03d}.json', 'w') as f:
                json.dump(ckpt, f, indent=2)
            print(f'  [CHECKPOINT] {PROXY_CHECKPOINT_DIR}/checkpoint_gen_{gen:03d}.json')

    print(f'\n  Done. Pareto={len(archive.front)} | Proxy calls={len(cache)} | Boosts={total_boosts}')

    # [4/4] Save results
    print('[4/4] Saving proxy Pareto front...')
    output = {'pareto_front': archive.front, 'history': history,
              'wild_type_proxy_cost': wt_proxy,
              'objective': 'physicochemical_proxy',
              'proxy_components': 'instability_index + GRAVY (Yang et al. 2024 style)'}
    with open(PROXY_OUTPUT_PATH, 'w') as f:
        json.dump(output, f, indent=2)

    print()
    print('=' * 72)
    print('  PHASE 5 PROXY RUN COMPLETE')
    print('=' * 72)
    print(f'  Pareto front candidates: {len(archive.front)}')
    print(f'  Wild-type proxy cost:    {wt_proxy:.4f}')
    best_s = max(archive.front, key=lambda c: c["structural_score"])
    best_c = min(archive.front, key=lambda c: c["mpb_score"])
    print(f'  Best structural:  {best_s["structural_score"]:.4f} -> {best_s["sequence"]}')
    print(f'  Best proxy cost:  {best_c["mpb_score"]:.4f} -> {best_c["sequence"]}')
    print(f'  Output: {PROXY_OUTPUT_PATH}')
    print('=' * 72)
    return archive, history


def _compute_structural_score(seq, feasibility_map, mutable_positions, wild_type):
    """Reuse Phase 4 structural scoring via feasibility map log-probs."""
    score = 0.0
    for pos in mutable_positions:
        key = f'pos_{pos}'
        if key not in feasibility_map:
            continue
        aa  = seq[pos]
        wt_aa = wild_type[pos]
        probs = feasibility_map[key]['probabilities']
        p     = probs.get(aa, 1e-9)
        p_wt  = probs.get(wt_aa, 1e-9)
        score += np.log(p) - np.log(p_wt)
    return float(score)


def _random_feasible(wild_type, mutable_positions, feasibility_map):
    """Sample a random feasible sequence using the ProteinMPNN prior."""
    seq = list(wild_type)
    for pos in mutable_positions:
        key   = f'pos_{pos}'
        probs = feasibility_map[key]['probabilities']
        aas   = list(probs.keys())
        ps    = np.array([probs[a] for a in aas])
        ps   /= ps.sum()
        seq[pos] = np.random.choice(aas, p=ps)
    return ''.join(seq)


def _sequence_entropy(population, mutable_positions):
    """Mean per-position Shannon entropy across the population."""
    entropies = []
    for pos in mutable_positions:
        counts = {}
        for c in population:
            aa = c['sequence'][pos]
            counts[aa] = counts.get(aa, 0) + 1
        n = len(population)
        h = -sum((v/n) * np.log2(v/n) for v in counts.values() if v > 0)
        entropies.append(h)
    return float(np.mean(entropies))


print(' run_phase5_proxy.py defined.')


In [ ]:
#  RUN
proxy_archive, proxy_history = execute_pipeline_phase_5_proxy()


In [ ]:
"""
divergence_analyzer.py
----
Phase 5.3: FBA vs Proxy Divergence Analysis.

Compares the FBA Pareto front (Phase 4 standard) against the proxy
Pareto front (Phase 5) to determine whether the FBA objective selects
for meaningfully different sequences - specifically whether it
systematically penalises metabolically expensive amino acids
(W, Y, F, M, H, C, R) in a way that physicochemical proxies do not.

This is the central empirical result of the BioForge framework paper.
"""

import json
import numpy as np
from scipy import stats
from collections import Counter

AA_BIOSYNTHETIC_COST = {
    'A':  11.7, 'R':  27.3, 'N':  14.7, 'D':  12.8, 'C':  24.7,
    'Q':  16.3, 'E':  15.3, 'G':  11.7, 'H':  38.8, 'I':  32.7,
    'L':  27.3, 'K':  30.3, 'M':  34.3, 'F':  52.0, 'P':  20.3,
    'S':  18.7, 'T':  18.7, 'W':  74.3, 'Y':  50.0, 'V':  23.3,
}
EXPENSIVE_AA  = {'W', 'Y', 'F', 'M', 'H', 'C', 'R'}
WILD_TYPE     = 'GFGCNGPWDEDDMQCHNHCKSIKGYKGGYCAKGGFVCKCY'


def run_divergence_analysis():
    print('=' * 70)
    print('  PHASE 5 - DIVERGENCE ANALYSIS: FBA vs Physicochemical Proxy')
    print('=' * 70)

    # Load both fronts
    with open('./pipeline_artifacts/phase4_pareto_front.json') as f:
        fba_data  = json.load(f)
    with open('./pipeline_artifacts/phase5_proxy_pareto_front.json') as f:
        proxy_data = json.load(f)

    fba_front   = fba_data.get('pareto_front', fba_data if isinstance(fba_data, list) else [])
    proxy_front = proxy_data.get('pareto_front', proxy_data if isinstance(proxy_data, list) else [])

    fba_seqs   = {c['sequence'] for c in fba_front}
    proxy_seqs = {c['sequence'] for c in proxy_front}

    print(f'\n  FBA front size   : {len(fba_front)}')
    print(f'  Proxy front size : {len(proxy_front)}')

    # - 1. Sequence Overlap -
    print('\n' + '=' * 70)
    print('CHECK 1: Sequence Overlap (Jaccard Similarity)')
    print('=' * 70)
    intersection = fba_seqs & proxy_seqs
    union        = fba_seqs | proxy_seqs
    jaccard      = len(intersection) / len(union) if union else 0
    only_fba     = fba_seqs   - proxy_seqs
    only_proxy   = proxy_seqs - fba_seqs
    print(f'  Sequences in both fronts      : {len(intersection)}')
    print(f'  Sequences only in FBA front   : {len(only_fba)}')
    print(f'  Sequences only in proxy front : {len(only_proxy)}')
    print(f'  Jaccard similarity            : {jaccard:.4f}  (0=no overlap, 1=identical)')
    if jaccard < 0.2:
        print('   LOW overlap - the two objectives select substantially different sequences.')
    elif jaccard < 0.5:
        print('  ~ MODERATE overlap - objectives partially agree but diverge meaningfully.')
    else:
        print('   HIGH overlap - objectives largely agree; divergence may be weak.')

    # - 2. Expensive AA Analysis -
    print('\n' + '=' * 70)
    print('CHECK 2: Expensive Amino Acid Content (W, Y, F, M, H, C, R)')
    print('=' * 70)

    def expensive_count(seq):
        return sum(1 for aa in seq if aa in EXPENSIVE_AA)

    def true_cost(seq):
        return sum(AA_BIOSYNTHETIC_COST.get(aa, 20.0) for aa in seq)

    fba_exp   = [expensive_count(c['sequence']) for c in fba_front]
    proxy_exp = [expensive_count(c['sequence']) for c in proxy_front]
    fba_cost_true   = [true_cost(c['sequence']) for c in fba_front]
    proxy_cost_true = [true_cost(c['sequence']) for c in proxy_front]

    wt_exp  = expensive_count(WILD_TYPE)
    wt_cost = true_cost(WILD_TYPE)

    print(f'  Wild-type: {wt_exp} expensive AAs | true biosyn cost: {wt_cost:.1f} ATP eq')
    print(f'\n  FBA front   - mean expensive AAs: {np.mean(fba_exp):.2f} (±{np.std(fba_exp):.2f})')
    print(f'                mean true cost:     {np.mean(fba_cost_true):.1f} ATP eq (±{np.std(fba_cost_true):.1f})')
    print(f'\n  Proxy front - mean expensive AAs: {np.mean(proxy_exp):.2f} (±{np.std(proxy_exp):.2f})')
    print(f'                mean true cost:     {np.mean(proxy_cost_true):.1f} ATP eq (±{np.std(proxy_cost_true):.1f})')

    # Mann-Whitney U test
    u_stat, p_val = stats.mannwhitneyu(fba_exp, proxy_exp, alternative='less')
    print(f'\n  Mann-Whitney U (FBA < Proxy on expensive AA count):')
    print(f'    U={u_stat:.1f}  p={p_val:.4f}')
    if p_val < 0.05:
        print('   SIGNIFICANT: FBA front has statistically fewer expensive AAs than proxy front.')
        print('    This is the key empirical result - FBA sees what proxies miss.')
    else:
        print('   Not significant at p<0.05 - expensive AA distributions overlap.')

    # - 3. Per-AA composition comparison -
    print('\n' + '=' * 70)
    print('CHECK 3: Per-AA Frequency Shift (FBA vs Proxy, mutable positions only)')
    print('=' * 70)
    with open('./pipeline_artifacts/phase2_mask.json') as f:
        mask = json.load(f)
    mutable_pos = sorted(int(k.split('_')[1]) for k, v in mask.items() if v['status'] == 'MUTABLE')

    def aa_freq(front):
        counts = Counter()
        total  = 0
        for c in front:
            for pos in mutable_pos:
                counts[c['sequence'][pos]] += 1
                total += 1
        return {aa: counts[aa]/total for aa in 'ACDEFGHIKLMNPQRSTVWY'}

    fba_freq   = aa_freq(fba_front)
    proxy_freq = aa_freq(proxy_front)

    print(f'  {"AA":<4} {"FBA freq":>10} {"Proxy freq":>12} {"Δ (FBA-Proxy)":>14} {"Biosyn cost":>12}')
    print(f'  {"-"*58}')
    diffs = []
    for aa in sorted('ACDEFGHIKLMNPQRSTVWY', key=lambda a: -AA_BIOSYNTHETIC_COST[a]):
        d    = fba_freq[aa] - proxy_freq[aa]
        flag = '  expensive' if aa in EXPENSIVE_AA else ''
        diffs.append((aa, d))
        print(f'  {aa:<4} {fba_freq[aa]:>10.4f} {proxy_freq[aa]:>12.4f} '
              f'{d:>+14.4f}  ({AA_BIOSYNTHETIC_COST[aa]:>5.1f} ATP){flag}')

    # - 4. Proxy-FBA cost correlation -
    print('\n' + '=' * 70)
    print('CHECK 4: Correlation Between FBA Cost and True Biosynthetic Cost')
    print('=' * 70)
    all_seqs  = list({c['sequence'] for c in fba_front + proxy_front})
    all_fba   = [next((c['mpb_score'] for c in fba_front if c['sequence'] == s), None)
                 for s in all_seqs]
    all_true  = [true_cost(s) for s in all_seqs]
    all_proxy = [next((c['mpb_score'] for c in proxy_front if c['sequence'] == s), None)
                 for s in all_seqs]

    # FBA vs true cost correlation
    pairs_fba = [(f, t) for f, t in zip(all_fba, all_true) if f is not None]
    if len(pairs_fba) >= 3:
        r_fba, p_fba = stats.pearsonr([p[0] for p in pairs_fba], [p[1] for p in pairs_fba])
        print(f'  FBA MPB vs true biosyn cost: r={r_fba:.4f}  p={p_fba:.4f}')
        print(f'    → FBA cost {"correlates" if abs(r_fba) > 0.5 else "does NOT strongly correlate"} with true ATP cost')

    # Proxy vs true cost correlation
    pairs_proxy = [(p, t) for p, t in zip(all_proxy, all_true) if p is not None]
    if len(pairs_proxy) >= 3:
        r_prx, p_prx = stats.pearsonr([p[0] for p in pairs_proxy], [p[1] for p in pairs_proxy])
        print(f'  Proxy cost vs true biosyn cost: r={r_prx:.4f}  p={p_prx:.4f}')
        print(f'    → Proxy cost {"correlates" if abs(r_prx) > 0.5 else "does NOT strongly correlate"} with true ATP cost')

    # - 5. Sequences unique to proxy front (the blind spots) -
    print('\n' + '=' * 70)
    print('CHECK 5: Sequences in Proxy Front NOT in FBA Front (Proxy Blind Spots)')
    print('=' * 70)
    print('  These are candidates the proxy rated highly but FBA would reject')
    print('  as metabolically expensive. They reveal what proxies miss.')
    print()
    blind_spots = [c for c in proxy_front if c['sequence'] in only_proxy]
    blind_spots.sort(key=lambda c: true_cost(c['sequence']), reverse=True)
    for c in blind_spots[:5]:
        tc   = true_cost(c['sequence'])
        ec   = expensive_count(c['sequence'])
        exp_aas = [aa for aa in c['sequence'] if aa in EXPENSIVE_AA]
        print(f'  {c["sequence"]}')
        print(f'    proxy_cost={c["mpb_score"]:.4f}  true_cost={tc:.1f} ATP  '
              f'expensive_AAs={ec} ({" ".join(exp_aas)})')
        print()

    # - Summary -
    print('=' * 70)
    print('  DIVERGENCE ANALYSIS - SUMMARY')
    print('=' * 70)
    print(f'  Jaccard similarity          : {jaccard:.4f}')
    print(f'  FBA   mean expensive AAs    : {np.mean(fba_exp):.2f}')
    print(f'  Proxy mean expensive AAs    : {np.mean(proxy_exp):.2f}')
    print(f'  FBA   mean true cost        : {np.mean(fba_cost_true):.1f} ATP eq')
    print(f'  Proxy mean true cost        : {np.mean(proxy_cost_true):.1f} ATP eq')
    print(f'  Mann-Whitney p-value        : {p_val:.4f}')
    print(f'  Proxy blind spots (n)       : {len(only_proxy)}')
    print()
    if p_val < 0.05 and jaccard < 0.5:
        print('   STRONG DIVERGENCE: FBA objective selects cheaper sequences')
        print('    in ways that physicochemical proxies cannot replicate.')
        print('    This validates the FBA-as-objective framework contribution.')
    elif p_val < 0.1 or jaccard < 0.3:
        print('  ~ MODERATE DIVERGENCE: Some evidence FBA and proxy differ.')
        print('    Report the direction and magnitude honestly.')
    else:
        print('   WEAK DIVERGENCE: FBA and proxy select similar sequences.')
        print('    The framework contribution claim is weakened. Consider')
        print('    refining the FBA objective or switching to shadow-price scoring.')
    print('=' * 70)

    # Save summary
    summary = {
        'jaccard_similarity':      jaccard,
        'fba_mean_expensive_aa':   float(np.mean(fba_exp)),
        'proxy_mean_expensive_aa': float(np.mean(proxy_exp)),
        'fba_mean_true_cost':      float(np.mean(fba_cost_true)),
        'proxy_mean_true_cost':    float(np.mean(proxy_cost_true)),
        'mannwhitney_p':           float(p_val),
        'n_only_fba':              len(only_fba),
        'n_only_proxy':            len(only_proxy),
        'n_intersection':          len(intersection),
        'aa_freq_fba':             fba_freq,
        'aa_freq_proxy':           proxy_freq,
    }
    with open('./pipeline_artifacts/phase5_divergence_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print('  Saved: ./pipeline_artifacts/phase5_divergence_summary.json')


print(' divergence_analyzer.py defined.')


In [ ]:
#  RUN
run_divergence_analysis()
